# RAG: генерация с дополнением из базы знаний

## 1. Цель этапа

На предыдущих этапах были отдельно подготовлены:

- базовая модель `Qwen/Qwen2.5-3B-Instruct`;
- улучшенный медицинский промпт;
- база знаний из клинических рекомендаций;
- dense retrieval на основе BGE.

На этом этапе retrieval и генерация объединяются в единый RAG-пайплайн.

Основное сравнение:

- **B:** Base Qwen + improved prompt;
- **C:** Base Qwen + improved prompt + RAG.

Цель этапа — проверить, как добавление релевантного контекста из клинических
рекомендаций влияет на:

- медицинскую корректность ответа;
- обоснованность ответа найденными источниками;
- количество неподтверждённых медицинских утверждений;
- корректность использования ссылок на источники;
- поведение модели, когда база знаний не содержит достаточной информации.

QLoRA на этом этапе не используется.

## 2. Архитектура RAG

RAG-пайплайн имеет следующий вид:

вопрос пользователя  
- embedding запроса  
- dense retrieval  
- top-k наиболее релевантных фрагментов  
- формирование контекста  
- RAG-промпт  
- Qwen  
- ответ с опорой на найденные источники

На этом этапе конфигурация retrieval фиксирована и повторно не подбирается.

Используется BGE dense retrieval, выбранный по результатам предыдущего
эксперимента.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

In [ ]:
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

MATERIALS_DIR = PROJECT_ROOT / "materials"
RESULTS_DIR = PROJECT_ROOT / "results"

RETRIEVAL_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "retrieval"
)

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print(
    "retrieval.py exists:",
    (
        PROJECT_ROOT
        / "src"
        / "retrieval.py"
    ).exists(),
)

In [3]:
from src.retrieval import (
    load_retrieval_config,
    load_retrieval_data,
    load_embedding_model,
    dense_search,
)

In [4]:
retrieval_config = load_retrieval_config(
    RETRIEVAL_DATA_DIR
)

chunks_df, document_embeddings = (
    load_retrieval_data(
        RETRIEVAL_DATA_DIR
    )
)

device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embedding_model = load_embedding_model(
    model_name=retrieval_config["embedding_model"],
    device=device,
)

print("Device:", device)
print("Chunks:", len(chunks_df))
print(
    "Embeddings:",
    document_embeddings.shape,
)
print(
    "Embedding model:",
    retrieval_config["embedding_model"],
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7655.81it/s]


Device: cuda
Chunks: 2135
Embeddings: (2135, 768)
Embedding model: BAAI/bge-base-en-v1.5


In [5]:
test_query = (
    "How should hyperkalemia be managed "
    "in chronic kidney disease?"
)

retrieved_chunks = dense_search(
    query=test_query,
    chunks_df=chunks_df,
    document_embeddings=document_embeddings,
    embedding_model=embedding_model,
    query_prefix=retrieval_config["query_prefix"],
    top_k=3,
)

retrieved_chunks[
    [
        "score",
        "document_title",
        "section_path",
        "page",
        "text",
    ]
]

,score,document_title,section_path,page,text
0,0.801014,VA/DOD Clinical Practice Guideline for the Pri...,Appendix M. Management of Hyperkalemia > D. Di...,147,for the Evaluation and Management of Chronic K...
1,0.798880,VA/DOD Clinical Practice Guideline for the Pri...,Appendix M. Management of Hyperkalemia > C. Ma...,146,upper limit of normal range They acknowledge t...
2,0.793596,VA/DOD Clinical Practice Guideline for the Pri...,Appendix M. Management of Hyperkalemia > C. Ma...,146,The aggressiveness of treatment of hyperkalemi...


## 3. Формирование контекста для RAG

Retriever возвращает отдельные фрагменты текста вместе с metadata.

Перед передачей в LLM найденные фрагменты объединяются в единый
структурированный контекст.

Для каждого фрагмента сохраняются:

- номер источника внутри промпта;
- название документа;
- раздел документа;
- страница;
- текст фрагмента.

Similarity score в промпт не передаётся. Он используется retrieval-системой
для ранжирования результатов, но сам по себе не является медицинской
информацией для генерации ответа.

In [6]:
def format_retrieval_context(retrieved_chunks):
    context_parts = []

    for source_number, (_, row) in enumerate(
        retrieved_chunks.iterrows(),
        start=1,
    ):
        source_block = (
            f"[Source {source_number}]\n"
            f"Document: {row['document_title']}\n"
            f"Section: {row['section_path']}\n"
            f"Page: {row['page']}\n"
            f"Text: {row['text']}"
        )

        context_parts.append(source_block)

    return "\n\n".join(context_parts)

## 4. Формирование RAG-промпта

Для сравнения вариантов B и C основной `improved prompt` остаётся тем же,
что использовался в Notebook 02.

В варианте C дополнительно используются:

- найденный контекст из клинических рекомендаций;
- инструкция опираться только на предоставленные источники;
- требование указывать ссылки в формате `[Source N]`;
- запрет дополнять ответ медицинскими утверждениями, которых нет
  в найденном контексте.

Таким образом:

- B получает improved prompt и вопрос пользователя;
- C получает тот же improved prompt, найденный контекст и дополнительные
  RAG-инструкции.

Поэтому сравнение B и C оценивает RAG-конфигурацию целиком, а не только
изолированный эффект retrieval.

In [7]:
IMPROVED_SYSTEM_PROMPT = """
You are a medical assistant providing general health information.

Use only the information given by the patient and medical knowledge you are confident about.

When answering:

1. Answer the main question directly.
2. Separate known facts from possible explanations.
3. If a diagnosis, laboratory result, vital sign, or medication effect cannot be interpreted confidently from the available information, explicitly say that rather than guessing.
4. Do not invent diagnoses, mechanisms, symptoms, test results, drug names, or medical history.
5. Do not introduce specific tests, treatments, or medication changes unless they are clearly necessary to answer the question.
6. Never advise starting, stopping, or changing medication without clinician supervision.
7. Mention urgent evaluation only when the information provided reasonably suggests a time-sensitive risk.
8. Keep the answer concise, practical, and appropriately cautious.

When important information is missing, state what is uncertain instead of filling the gap with assumptions.
""".strip()

In [8]:
RAG_SYSTEM_INSTRUCTION = """
For this RAG condition, the rule above allowing the use of medical knowledge
you are confident about is overridden.

For this RAG answer, the retrieved guideline excerpts are the only allowed
evidence for medical factual claims.

Do not add medical facts from your own background knowledge unless they are
explicitly supported by the retrieved excerpts.

Preserve the meaning, direction, and strength of guideline recommendations
exactly as stated in the retrieved evidence.

If a source states that there is insufficient evidence to recommend for or
against an intervention, do not convert it into a positive or negative
recommendation.

Do not strengthen or weaken recommendations such as "recommend", "suggest",
"weak for", "weak against", "neither for nor against", or similar wording.

Every medical recommendation, threshold, treatment statement, diagnostic
statement, or other factual medical claim must be followed by at least one
citation in the exact format [Source N].

Place each citation immediately after the claim it supports rather than
grouping unrelated citations at the end of the answer.

Use only source numbers that appear in the retrieved context.

Do not cite a source unless that source directly supports the corresponding
claim.

If different retrieved sources provide different types or strengths of
recommendations, keep those distinctions in the answer rather than combining
them into one general recommendation.

If the retrieved excerpts do not contain enough evidence to answer part of
the question, explicitly state that the available guideline context is
insufficient for that part instead of filling the gap from memory.

Do not invent missing recommendations, explanations, thresholds, medication
details, or interpretations that are not supported by the retrieved context.
""".strip()

def build_rag_messages(
    question,
    context,
    system_prompt=IMPROVED_SYSTEM_PROMPT,
):
    rag_system_prompt = (
        system_prompt
        + "\n\n"
        + RAG_SYSTEM_INSTRUCTION
    )

    user_message = f"""
Retrieved guideline context:

{context}

Question:

{question}

Answer the question using only the retrieved evidence.
Cite each medical factual claim with [Source N].
""".strip()

    return [
        {
            "role": "system",
            "content": rag_system_prompt,
        },
        {
            "role": "user",
            "content": user_message,
        },
    ]

In [9]:
rag_context = format_retrieval_context(
    retrieved_chunks
)

print(rag_context)

[Source 1]
Document: VA/DOD Clinical Practice Guideline for the Primary Care Management of Chronic Kidney Disease
Section: Appendix M. Management of Hyperkalemia > D. Dietary Considerations
Page: 147
Text: for the Evaluation and Management of Chronic Kidney Disease (3) recommends, “Provide advice to limit the intake of foods rich in bioavailable potassium (e.g., processed foods) for people with CKD G3-G5 who have a history of hyperkalemia.” The involvement of a renal dietitian can be helpful, as can the use of teaching materials that emphasize the avoidance of processed foods with potassium additives (e.g., Potassium Management in Kidney Disease).

[Source 2]
Document: VA/DOD Clinical Practice Guideline for the Primary Care Management of Chronic Kidney Disease
Section: Appendix M. Management of Hyperkalemia > C. Management
Page: 146
Text: upper limit of normal range They acknowledge that it is not known whether EKG changes are sensitive in the prediction of potentially lethal arrhythmi

In [10]:
rag_messages = build_rag_messages(
    question=test_query,
    context=rag_context,
)

for message in rag_messages:
    print(
        "\n",
        "=" * 80,
        f"\nROLE: {message['role']}\n",
    )
    print(message["content"])


ROLE: system

You are a medical assistant providing general health information.

Use only the information given by the patient and medical knowledge you are confident about.

When answering:

1. Answer the main question directly.
2. Separate known facts from possible explanations.
3. If a diagnosis, laboratory result, vital sign, or medication effect cannot be interpreted confidently from the available information, explicitly say that rather than guessing.
4. Do not invent diagnoses, mechanisms, symptoms, test results, drug names, or medical history.
5. Do not introduce specific tests, treatments, or medication changes unless they are clearly necessary to answer the question.
6. Never advise starting, stopping, or changing medication without clinician supervision.
7. Mention urgent evaluation only when the information provided reasonably suggests a time-sensitive risk.
8. Keep the answer concise, practical, and appropriately cautious.

When important information is missing, state what

## 5. Первая генерация ответа с RAG

Для генерации используется та же базовая модель и те же параметры генерации,
что и в Notebook 02.

Это позволяет сравнивать:

- B — Base + improved prompt;
- C — Base + improved prompt + RAG.

Модель, квантизация и параметры генерации остаются одинаковыми.
В варианте C дополнительно используются найденный контекст и инструкции
по работе с ним.

In [11]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

In [13]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
)

model.eval()

W0914 18:55:46.783000 13676 .venv-med-assist\Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 434/434 [00:02<00:00, 169.08it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear4bit(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm

In [14]:
print("Model:", MODEL_NAME)
print("Model device:", model.device)
print("Tokenizer:", type(tokenizer).__name__)
print("Model:", type(model).__name__)

Model: Qwen/Qwen2.5-3B-Instruct
Model device: cuda:0
Tokenizer: Qwen2Tokenizer
Model: Qwen2ForCausalLM


In [15]:
def generate_from_messages(
    messages,
    model,
    tokenizer,
    max_new_tokens=512,
):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = generated_ids[
        :,
        model_inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.batch_decode(
        new_tokens,
        skip_special_tokens=True,
    )[0]

    return answer.strip()

In [16]:
rag_answer = generate_from_messages(
    messages=rag_messages,
    model=model,
    tokenizer=tokenizer,
)

print(rag_answer)

In the management of hyperkalemia in chronic kidney disease (CKD), the following steps are recommended:

First-line:
- Review non-RAASi culprit medications and discontinue when possible.
- Consider appropriate moderation of dietary potassium intake.

Second-line:
- Use diuretics (when appropriate and risk of volume depletion is low).
- For outpatients with acute hyperkalemia who have a potassium concentration of >6.0 mmol/L or hyperkalemia with any new EKG changes, refer them to a facility with cardiac monitoring, usually an emergency department that can address this urgently.

The aggressiveness of treatment depends on the degree of elevation and the presence or absence of EKG findings. Acute hyperkalemia is classified as mild, moderate, or severe based on the potassium concentration and the presence or absence of EKG changes:
- Mild: 5.0-5.9 mmol/L without EKG changes
- Moderate: 5.0-5.9 mmol/L with EKG changes or 6.0-6.4 mmol/L without EKG changes
- Severe: 6.0-6.4 mmol/L with EKG c

### Выбор числа найденных фрагментов

Для RAG используется `top_k = 3`.

В предыдущей оценке dense retrieval значения section-level Hit@3 и Hit@5
совпали и составили `0.936`.

Следовательно, увеличение контекста с трёх до пяти фрагментов не давало
дополнительного recall на выборке для разработки.

Поэтому `top_k = 3` позволяет сохранить наблюдаемое качество retrieval
при меньшем объёме контекста.

После этого параметр фиксируется и не изменяется по результатам дальнейшей
оценки RAG.

In [17]:
TOP_K = 3


def run_rag(question, top_k=TOP_K):
    retrieved = dense_search(
        query=question,
        chunks_df=chunks_df,
        document_embeddings=document_embeddings,
        embedding_model=embedding_model,
        query_prefix=retrieval_config["query_prefix"],
        top_k=top_k,
    )

    context = format_retrieval_context(
        retrieved
    )

    messages = build_rag_messages(
        question=question,
        context=context,
    )

    answer = generate_from_messages(
        messages=messages,
        model=model,
        tokenizer=tokenizer,
    )

    return {
        "question": question,
        "retrieved_chunks": retrieved,
        "context": context,
        "answer": answer,
    }

In [18]:
import re


def extract_source_citations(answer):
    citations = re.findall(
        r"\[Source\s+(\d+)\]",
        answer,
    )

    return sorted(
        {int(source_id) for source_id in citations}
    )

In [19]:
extract_source_citations(rag_answer)

[1, 2, 3]

In [20]:
probe_questions = [
    "How should asthma be diagnosed in adults?",
    "How should chronic low back pain be managed?",
    "How should hypertension be treated in adults?",
    "How should glycemic control be assessed in type 2 diabetes?",
]

In [21]:
probe_result = run_rag(
    probe_questions[0]
)

print("QUESTION:")
print(probe_result["question"])

print("\nRETRIEVED:")
display(
    probe_result["retrieved_chunks"][
        [
            "score",
            "document_title",
            "section_path",
            "page",
        ]
    ]
)

print("\nANSWER:")
print(probe_result["answer"])

print("\nCITATIONS:")
print(
    extract_source_citations(
        probe_result["answer"]
    )
)

QUESTION:
How should asthma be diagnosed in adults?

RETRIEVED:


,score,document_title,section_path,page
0,0.720767,VA/DOD Clinical Practice Guideline for the Pri...,IX. Recommendations > B. Treatment and Management,57
1,0.713335,VA/DOD Clinical Practice Guideline for the Pri...,IX. Recommendations > B. Treatment and Management,57
2,0.708747,VA/DOD Clinical Practice Guideline for the Pri...,II. Background > A. Description of Asthma,7



ANSWER:
Asthma in adults should be diagnosed based on a clinical diagnosis based on history, physical examination, and findings suggestive of airway hyperactivity. [Source 1] Airway inflammation and bronchial hyperreactivity are considered the primary underlying pathologic processes. [Source 3] Despite these unifying characteristics, asthma is a very heterogeneous condition, and the diagnosis of asthma is not solely dependent on spirometry. [Source 1] Objective measurements of airway reactivity may be helpful but are not essential for the diagnosis. [Source 1] Routine spirometry for monitoring patients with stable asthma is not recommended due to lack of evidence showing significant improvement in patient outcomes. [Source 1] Spirometry may be used in specific cases, such as in active-duty military members, but its routine use is not supported by current literature. [Source 1]

CITATIONS:
[1, 3]


In [22]:
for i, row in probe_result["retrieved_chunks"].iterrows():
    print("=" * 100)
    print(f"SOURCE {i + 1}")
    print("Score:", row["score"])
    print("Section:", row["section_path"])
    print("Page:", row["page"])
    print()
    print(row["text"])
    print()

SOURCE 1
Score: 0.7207670211791992
Section: IX. Recommendations > B. Treatment and Management
Page: 57

to mitigate concerns related to resources and stigma. Thus, the Work Group decided upon a Weak for recommendation. d. Monitoring and Follow-up Recommendation 18. We suggest against utilizing spirometry for routine monitoring of patients with stable asthma. (Weak against | Not reviewed, Not changed) Discussion The diagnosis of asthma is a clinical diagnosis based on history, physical examination, and findings suggestive of airway hyperactivity. While objective measurements of airway reactivity (specifically reversible obstruction post-bronchodilator) may be helpful in the diagnosis of asthma, the lack of objective reversibility does not disqualify the diagnosis. Furthermore, the use of spirometry in routine monitoring of patients with asthma was not found to significantly improve patient outcomes on the standardized ACT. An RCT by Oei et al. 2011 demonstrated no statistically signific

In [23]:
remaining_probe_questions = [
    "How should chronic low back pain be managed?",
    "How should hypertension be treated in adults?",
    "How should glycemic control be assessed in type 2 diabetes?",
]

In [24]:
def inspect_rag_result(result):
    print("=" * 100)
    print("QUESTION:")
    print(result["question"])

    print("\nRETRIEVED:")
    display(
        result["retrieved_chunks"][
            [
                "score",
                "document_title",
                "section_path",
                "page",
            ]
        ]
    )

    print("\nANSWER:")
    print(result["answer"])

    print("\nCITATIONS:")
    print(
        extract_source_citations(
            result["answer"]
        )
    )

In [25]:
probe_lbp = run_rag(
    remaining_probe_questions[0]
)

inspect_rag_result(
    probe_lbp
)

QUESTION:
How should chronic low back pain be managed?

RETRIEVED:


,score,document_title,section_path,page
0,0.702335,VA/DoD Clinical Practice Guideline for the Dia...,IX. Recommendations,23
1,0.697911,VA/DoD Clinical Practice Guideline for the Dia...,VII. Approach to Care in Department of Veteran...,16
2,0.689820,VA/DoD Clinical Practice Guideline for the Dia...,VIII. Algorithm > B. Module B: Management of L...,22



ANSWER:
For patients with chronic low back pain, the VA/DoD Clinical Practice Guideline recommends the following:

- We suggest cognitive behavioral therapy. [Source 1]
- We suggest a structured clinician-directed exercise program (e.g., aerobic, aquatic, mechanical diagnosis and therapy, mobility, motor control, Pilates, strengthening exercises, structured walking program, tai chi). [Source 1]
- We suggest spinal mobilization/manipulation. [Source 1]

For other interventions, the guideline neither recommends nor suggests them:
- Pain neuroscience education, clinician-directed education with patient-led goal setting, or back school. [Source 1]
- Technology-based modalities. [Source 1]
- Mindfulness-based stress reduction. [Source 1]
- Lumbar supports. [Source 1]

The guideline also notes that managing low back pain in patients with co-occurring conditions requires collaboration with other care providers and may involve early specialist consultation. [Source 2]

CITATIONS:
[1, 2]


In [26]:
probe_hypertension = run_rag(
    remaining_probe_questions[1]
)

inspect_rag_result(
    probe_hypertension
)

QUESTION:
How should hypertension be treated in adults?

RETRIEVED:


,score,document_title,section_path,page
0,0.790755,Guideline for the pharmacological treatment of...,6 Implementation tools > 6.1 Guideline recomme...,27
1,0.790393,Guideline for the pharmacological treatment of...,6 Implementation tools > 6.1 Guideline recomme...,26
2,0.777983,Guideline for the pharmacological treatment of...,1 Introduction,2



ANSWER:
In the pharmacological treatment of hypertension in adults, the guidelines recommend treating adults with a systolic blood pressure (SBP) ≥130 mmHg or ≥90 mmHg (for those with CVD, DM, CKD), or SBP ≥140 mmHg, with medications from one of the following three classes: thiazide and thiazide-like agents, ACEi/ARB, and long-acting dihydropyridine CCB. 

The treatment targets are <140/90 mmHg (SBP <130 mmHg for high-risk patients with CVD, DM, CKD).

Initiation of pharmacological treatment should start no later than four weeks after the diagnosis of hypertension. If the BP level is high or there is evidence of end organ damage, treatment should be started without delay.

Basic laboratory testing (electrolytes, creatinine, lipogram, glucose, HbA1C, urine dipstick, and ECG) should occur as long as it does not delay treatment. A cardiovascular risk assessment can be conducted immediately (if it does not delay initiation of treatment) or at a later visit.

The choice between monotherapy

In [27]:
probe_t2d = run_rag(
    remaining_probe_questions[2]
)

inspect_rag_result(
    probe_t2d
)

QUESTION:
How should glycemic control be assessed in type 2 diabetes?

RETRIEVED:


,score,document_title,section_path,page
0,0.732655,VA/DoD Clinical Practice Guideline for the Man...,IX. Recommendations > C. Diabetes Mellitus,39
1,0.732650,VA/DoD Clinical Practice Guideline for the Man...,IX. Recommendations,25
2,0.729322,VA/DoD Clinical Practice Guideline for the Man...,IX. Recommendations > C. Diabetes Mellitus,38



ANSWER:
In assessing glycemic control in type 2 diabetes, glycemic variability can be used as an additional measure alongside average or cumulative values. Glycemic variability refers to the variation in glucose or HbA1c levels over time. It can be determined over various time intervals, such as between days, between visits, and includes measures like standard deviation (SD) and coefficient of variation (CV). [Source 3]

The benefits of using glycemic variability as a prognostic factor include its association with risk of hypoglycemia, morbidity, and mortality. However, the determinants of glycemic variability differ and can include factors such as glucose-lowering medication regimens, medication adherence, level of HbA1c, comorbidities, engagement with self-care, food insecurity, and financial and social support. [Source 3]

It is suggested that for adults with type 2 diabetes, high glycemic variability over time (e.g., fluctuation in HbA1c or fasting blood glucose) can be used as a 

### Проверка RAG-инструкций

Первоначальная версия RAG-промпта позволяла модели использовать одновременно
найденный контекст и собственные медицинские знания.

На диагностических примерах это привело к двум проблемам:

- модель добавляла сведения, отсутствующие в найденных источниках;
- требование использовать ссылки `[Source N]` выполнялось нестабильно.

Поэтому в системный промпт была добавлена отдельная RAG-инструкция,
запрещающая использовать фоновые медицинские знания для фактических
утверждений, если они не подтверждены найденным контекстом.

После изменения инструкции использование ссылок стало стабильнее.

При этом само наличие `[Source N]` ещё не означает, что источник действительно
подтверждает соответствующее утверждение. Корректность ссылок и
обоснованность ответа необходимо оценивать отдельно.

## 6. Проверка обоснованности ответа и корректности ссылок

Наличие `[Source N]` само по себе не гарантирует, что ответ действительно
основан на найденном контексте.

Для диагностических примеров отдельно проверяется:

1. использует ли модель только существующие номера источников;
2. подтверждает ли указанный источник соответствующее медицинское утверждение;
3. появились ли в ответе сведения, которых нет в найденном контексте;
4. сохранён ли исходный смысл рекомендации;
5. не изменены ли направление и сила рекомендации при генерации ответа.

Формат ссылок можно проверять автоматически.

Содержательную корректность ссылок и обоснованность ответа необходимо
оценивать отдельно по тексту retrieved evidence.

In [28]:
def check_citation_format(answer, n_sources):
    citations = extract_source_citations(answer)

    invalid_citations = [
        source_id
        for source_id in citations
        if source_id < 1 or source_id > n_sources
    ]

    return {
        "citations": citations,
        "has_citations": len(citations) > 0,
        "invalid_citations": invalid_citations,
        "all_source_ids_valid": len(invalid_citations) == 0,
    }

In [29]:
rag_messages = build_rag_messages(
    question=test_query,
    context=rag_context,
)

rag_answer = generate_from_messages(
    messages=rag_messages,
    model=model,
    tokenizer=tokenizer,
)

print(rag_answer)

In the management of hyperkalemia in chronic kidney disease (CKD), the following steps are recommended:

First-line:
- Review non-RAASi culprit medications and discontinue when possible.
- Consider appropriate moderation of dietary potassium intake.

Second-line:
- Use diuretics (when appropriate and risk of volume depletion is low).
- For outpatients with acute hyperkalemia who have a potassium concentration of >6.0 mmol/L or hyperkalemia with any new EKG changes, refer them to a facility with cardiac monitoring, usually an emergency department that can address this urgently.

The aggressiveness of treatment depends on the degree of elevation and the presence or absence of EKG findings. Acute hyperkalemia is classified as mild, moderate, or severe based on the potassium concentration and the presence or absence of EKG changes:
- Mild: 5.0-5.9 mmol/L without EKG changes
- Moderate: 5.0-5.9 mmol/L with EKG changes or 6.0-6.4 mmol/L without EKG changes
- Severe: 6.0-6.4 mmol/L with EKG c

In [30]:
check_citation_format(
    rag_answer,
    n_sources=len(retrieved_chunks),
)

{'citations': [1, 2, 3],
 'has_citations': True,
 'invalid_citations': [],
 'all_source_ids_valid': True}

In [31]:
def inspect_answer_evidence(result):
    print("=" * 100)
    print("QUESTION")
    print(result["question"])

    print("\n" + "=" * 100)
    print("ANSWER")
    print(result["answer"])

    for source_number, (_, row) in enumerate(
        result["retrieved_chunks"].iterrows(),
        start=1,
    ):
        print("\n" + "=" * 100)
        print(f"SOURCE {source_number}")
        print("Section:", row["section_path"])
        print("Page:", row["page"])
        print()
        print(row["text"])

In [32]:
inspect_answer_evidence(
    probe_lbp
)

QUESTION
How should chronic low back pain be managed?

ANSWER
For patients with chronic low back pain, the VA/DoD Clinical Practice Guideline recommends the following:

- We suggest cognitive behavioral therapy. [Source 1]
- We suggest a structured clinician-directed exercise program (e.g., aerobic, aquatic, mechanical diagnosis and therapy, mobility, motor control, Pilates, strengthening exercises, structured walking program, tai chi). [Source 1]
- We suggest spinal mobilization/manipulation. [Source 1]

For other interventions, the guideline neither recommends nor suggests them:
- Pain neuroscience education, clinician-directed education with patient-led goal setting, or back school. [Source 1]
- Technology-based modalities. [Source 1]
- Mindfulness-based stress reduction. [Source 1]
- Lumbar supports. [Source 1]

The guideline also notes that managing low back pain in patients with co-occurring conditions requires collaboration with other care providers and may involve early special

## 7. Сравнение Base + improved prompt и Base + improved prompt + RAG

После диагностических экспериментов RAG-конфигурация фиксируется.

Далее сравниваются две системы:

- **B — Base + improved prompt:** Qwen получает системный промпт и вопрос;
- **C — Base + improved prompt + RAG:** тот же Qwen дополнительно получает
  найденный контекст и RAG-инструкции.

В обеих системах одинаковы:

- модель `Qwen/Qwen2.5-3B-Instruct`;
- 4-bit quantization;
- основной `IMPROVED_SYSTEM_PROMPT`;
- `do_sample=False`;
- `max_new_tokens=512`.

Вариант C отличается от B двумя связанными компонентами:

- наличием найденного guideline-контекста;
- дополнительными инструкциями по использованию источников и ссылок.

Поэтому результаты B vs C характеризуют полный RAG-пайплайн и не могут
интерпретироваться как изолированный эффект одного retrieval.

In [33]:
def run_baseline(question):
    messages = [
        {
            "role": "system",
            "content": IMPROVED_SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    answer = generate_from_messages(
        messages=messages,
        model=model,
        tokenizer=tokenizer,
    )

    return {
        "question": question,
        "answer": answer,
    }

In [34]:
comparison_question = (
    "How should hyperkalemia be managed in chronic kidney disease?"
)

baseline_result = run_baseline(
    comparison_question
)

rag_result = run_rag(
    comparison_question
)

In [35]:
print("=" * 100)
print("QUESTION")
print(comparison_question)

print("\n" + "=" * 100)
print("B — BASE + IMPROVED PROMPT")
print(baseline_result["answer"])

print("\n" + "=" * 100)
print("C — BASE + IMPROVED PROMPT + RAG")
print(rag_result["answer"])

QUESTION
How should hyperkalemia be managed in chronic kidney disease?

B — BASE + IMPROVED PROMPT
Hyperkalemia in patients with chronic kidney disease (CKD) requires careful management because their kidneys may not effectively excrete potassium. Management typically includes monitoring serum potassium levels and adjusting dietary intake and medication as needed. It's important to avoid overusing diuretics, which can further reduce kidney function, and to monitor for signs of hyperkalemia such as muscle weakness, arrhythmias, and ECG changes. Consultation with a nephrologist is essential to tailor the management plan to the individual patient's needs.

C — BASE + IMPROVED PROMPT + RAG
In the management of hyperkalemia in chronic kidney disease (CKD), the following steps are recommended:

First-line:
- Review non-RAASi culprit medications and discontinue when possible.
- Consider appropriate moderation of dietary potassium intake.

Second-line:
- Use diuretics (when appropriate and risk

### Зафиксированная RAG-конфигурация

Для последующих экспериментов фиксируются:

- retriever: `BAAI/bge-base-en-v1.5`;
- dense retrieval по нормализованным embeddings;
- `top_k = 3`;
- сохранённый retrieval corpus из Notebook 03;
- `IMPROVED_SYSTEM_PROMPT`;
- `RAG_SYSTEM_INSTRUCTION`;
- модель `Qwen/Qwen2.5-3B-Instruct`;
- 4-bit quantization;
- детерминированная генерация (`do_sample=False`);
- `max_new_tokens=512`.

После этой точки RAG-промпт и параметры retrieval не изменяются
по результатам evaluation-вопросов.

### Проверка B и C на одном вопросе

Обе экспериментальные ветки работают.

Вариант B дал общий ответ, основанный преимущественно на внутренних знаниях
модели.

Вариант C использовал найденные guideline-фрагменты и дал более конкретный
ответ.

При этом уже на этом примере обнаружились проблемы grounded generation:

- ссылки были сгруппированы в конце ответа, а не размещены рядом
  с соответствующими утверждениями;
- часть структуры рекомендации была интерпретирована неточно.

После этой проверки RAG-промпт и retrieval-конфигурация больше не изменяются.

## 8. Генерация ответов B и C на выборке для разработки

Для сравнения B и C используются одни и те же вопросы.

Для каждого вопроса сохраняются:

- ответ B — Base + improved prompt;
- ответ C — Base + improved prompt + RAG;
- найденные источники для варианта C.

Эта выборка уже использовалась при разработке retrieval и анализе ошибок.

Поэтому результаты на ней рассматриваются как evaluation на этапе разработки,
а не как независимая финальная оценка.

In [36]:
import json

RETRIEVAL_EVAL_PATH = (
    MATERIALS_DIR
    / "retrieval_eval_v1.jsonl"
)

with open(
    RETRIEVAL_EVAL_PATH,
    "r",
    encoding="utf-8",
) as f:
    rag_dev_questions = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Questions:", len(rag_dev_questions))
print(rag_dev_questions[0])

Questions: 47
{'eval_id': 'dev_0005', 'candidate_id': 'candidate_0005', 'question_source': 'real_dev', 'query': 'Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?', 'relevant_document_id': 'va_dod_major_depression_2022', 'relevant_sections': ['VIII. Algorithm > A. Module A: Initial Assessment and Treatment', 'IX. Recommendations > C. Treatment Setting', 'IX. Recommendations > D. Treatment of Uncomplicated MDD', 'IX. Recommendations > H. Self-help, Complementary, and Alternative Treatments']}


### Формирование одной записи B vs C

Для каждого вопроса сохраняются не только ответы двух вариантов,
но и найденные источники варианта C.

Это позволяет отдельно оценивать:

- качество ответа B;
- качество ответа C;
- обоснованность ответа C;
- соответствие ссылок конкретным найденным фрагментам.

In [37]:
def serialize_retrieved_chunks(retrieved_chunks):
    sources = []

    for source_number, (_, row) in enumerate(
        retrieved_chunks.iterrows(),
        start=1,
    ):
        sources.append(
            {
                "source_id": source_number,
                "chunk_id": str(row["chunk_id"]),
                "document_id": str(row["document_id"]),
                "document_title": str(row["document_title"]),
                "section_path": str(row["section_path"]),
                "page": str(row["page"]),
                "score": float(row["score"]),
                "text": str(row["text"]),
            }
        )

    return sources

In [38]:
def generate_b_vs_c_record(item):
    question = item["query"]

    baseline_result = run_baseline(
        question
    )

    rag_result = run_rag(
        question
    )

    retrieved_sources = serialize_retrieved_chunks(
        rag_result["retrieved_chunks"]
    )

    rag_citations = extract_source_citations(
        rag_result["answer"]
    )

    citation_check = check_citation_format(
        rag_result["answer"],
        n_sources=len(retrieved_sources),
    )

    return {
        "eval_id": item["eval_id"],
        "candidate_id": item.get("candidate_id"),
        "question_source": item["question_source"],
        "question": question,

        "relevant_document_id": item[
            "relevant_document_id"
        ],
        "relevant_sections": item[
            "relevant_sections"
        ],

        "baseline_answer": baseline_result[
            "answer"
        ],
        "rag_answer": rag_result[
            "answer"
        ],

        "retrieved_sources": retrieved_sources,

        "rag_citations": rag_citations,
        "rag_has_citations": citation_check[
            "has_citations"
        ],
        "rag_all_source_ids_valid": citation_check[
            "all_source_ids_valid"
        ],
    }

In [39]:
test_comparison_record = generate_b_vs_c_record(
    rag_dev_questions[0]
)

In [40]:
print("=" * 100)
print("ID:")
print(test_comparison_record["eval_id"])

print("\nQUESTION:")
print(test_comparison_record["question"])

print("\n" + "=" * 100)
print("B — BASE + IMPROVED PROMPT")
print(test_comparison_record["baseline_answer"])

print("\n" + "=" * 100)
print("C — BASE + IMPROVED PROMPT + RAG")
print(test_comparison_record["rag_answer"])

print("\nCITATIONS:")
print(test_comparison_record["rag_citations"])

print("\nRETRIEVED SOURCES:")
for source in test_comparison_record[
    "retrieved_sources"
]:
    print(
        source["source_id"],
        source["document_id"],
        "→",
        source["section_path"],
    )

ID:
dev_0005

QUESTION:
Hi my boyfriend and I have been together for over a year and just recently (about 5 months after his best friends death) he has begun to act strange. Hes temper has become a lot worse and he has a short fuse. He is only 20. He feels so much stress and pressure even when there isnt things to stress about. Hes not himself and admitted that he is feeling depressed lately. His moods switch in an instant. Is there something i can do about it? Does he need medical help for depression or manic depression? Is there something besides medication to help?

B — BASE + IMPROVED PROMPT
It sounds like your boyfriend is experiencing significant emotional distress following the loss of his friend. His behavior change, including mood swings and irritability, could be signs of depression or other mental health issues. However, without a clinical evaluation, it's difficult to determine the exact cause.

Given his age and the recent loss, it's important to support him through this d

### Генерация ответов на выборке для разработки

RAG-конфигурация и промпт зафиксированы.

Все 47 вопросов последовательно прогоняются через две системы:

- B — Base + improved prompt;
- C — Base + improved prompt + RAG.

Результат каждого вопроса сохраняется сразу после генерации, чтобы
не потерять уже полученные ответы при возможной остановке notebook.

Эти результаты используются только для оценки на этапе разработки.

In [ ]:
RAG_DEV_RESULTS_PATH = (
    RESULTS_DIR
    / "rag_dev_b_vs_c_v1.jsonl"
)

print(RAG_DEV_RESULTS_PATH)

In [42]:
with open(
    RAG_DEV_RESULTS_PATH,
    "w",
    encoding="utf-8",
) as f:
    f.write(
        json.dumps(
            test_comparison_record,
            ensure_ascii=False,
        )
        + "\n"
    )

print("Saved:", test_comparison_record["eval_id"])

Saved: dev_0005


In [43]:
def load_completed_eval_ids(path):
    if not path.exists():
        return set()

    completed_ids = set()

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                completed_ids.add(
                    record["eval_id"]
                )

    return completed_ids

In [44]:
completed_ids = load_completed_eval_ids(
    RAG_DEV_RESULTS_PATH
)

print(
    f"Already completed: "
    f"{len(completed_ids)}/{len(rag_dev_questions)}"
)


for index, item in enumerate(
    rag_dev_questions,
    start=1,
):
    eval_id = item["eval_id"]

    if eval_id in completed_ids:
        continue

    print(
        f"[{index}/{len(rag_dev_questions)}] "
        f"{eval_id}"
    )

    try:
        record = generate_b_vs_c_record(
            item
        )

        with open(
            RAG_DEV_RESULTS_PATH,
            "a",
            encoding="utf-8",
        ) as f:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

        completed_ids.add(eval_id)

        print("saved")

    except Exception as error:
        print(
            f"ERROR in {eval_id}:",
            repr(error),
        )
        break

Already completed: 1/47
[2/47] dev_0014
saved
[3/47] dev_0029
saved
[4/47] dev_0048
saved
[5/47] dev_0078
saved
[6/47] dev_0097
saved
[7/47] dev_0135
saved
[8/47] who_01
saved
[9/47] who_02
saved
[10/47] who_03
saved
[11/47] who_04
saved
[12/47] who_05
saved
[13/47] t2d_01
saved
[14/47] t2d_02
saved
[15/47] t2d_03
saved
[16/47] t2d_04
saved
[17/47] t2d_05
saved
[18/47] ckd_01
saved
[19/47] ckd_02
saved
[20/47] ckd_03
saved
[21/47] ckd_04
saved
[22/47] ckd_05
saved
[23/47] asthma_01
saved
[24/47] asthma_02
saved
[25/47] asthma_03
saved
[26/47] asthma_04
saved
[27/47] asthma_05
saved
[28/47] lbp_01
saved
[29/47] lbp_02
saved
[30/47] lbp_03
saved
[31/47] lbp_04
saved
[32/47] lbp_05
saved
[33/47] mdd_01
saved
[34/47] mdd_02
saved
[35/47] mdd_03
saved
[36/47] mdd_04
saved
[37/47] mdd_05
saved
[38/47] pregnancy_01
saved
[39/47] pregnancy_02
saved
[40/47] pregnancy_03
saved
[41/47] pregnancy_04
saved
[42/47] pregnancy_05
saved
[43/47] sti_01
saved
[44/47] sti_02
saved
[45/47] sti_03
saved
[46

In [45]:
with open(
    RAG_DEV_RESULTS_PATH,
    "r",
    encoding="utf-8",
) as f:
    rag_dev_results = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print(
    "Generated:",
    len(rag_dev_results),
)

assert len(rag_dev_results) == len(
    rag_dev_questions
)

assert len({
    row["eval_id"]
    for row in rag_dev_results
}) == len(rag_dev_results)


Generated: 47


## 9. Автоматическая проверка результатов

Перед ручной оценкой выполняются проверки, которые можно вычислить
автоматически:

- наличие ссылок в RAG-ответе;
- использование только существующих номеров источников;
- попадание релевантного документа в top-3;
- попадание заранее размеченного раздела в top-3;
- длина ответов B и C.

Попадание в `relevant_sections` используется только как диагностическая
метрика.

Ранее было показано, что section-level gold-разметка неполна:
релевантный ответу текст может находиться и в другом разделе того же документа.

Эти проверки не измеряют медицинскую корректность ответа, корректность
ссылок по смыслу или groundedness.

In [46]:
def build_automatic_eval_record(record):
    retrieved_sources = record["retrieved_sources"]

    retrieved_document_ids = [
        source["document_id"]
        for source in retrieved_sources
    ]

    retrieved_sections = [
        source["section_path"]
        for source in retrieved_sources
    ]

    gold_document_hit = (
        record["relevant_document_id"]
        in retrieved_document_ids
    )

    gold_section_hit = any(
        section in record["relevant_sections"]
        for section in retrieved_sections
    )

    return {
        "eval_id": record["eval_id"],
        "question_source": record["question_source"],

        "rag_has_citations": record[
            "rag_has_citations"
        ],

        "rag_all_source_ids_valid": record[
            "rag_all_source_ids_valid"
        ],

        "rag_num_unique_citations": len(
            record["rag_citations"]
        ),

        "gold_document_hit_top3": gold_document_hit,
        "gold_section_hit_top3": gold_section_hit,

        "baseline_answer_words": len(
            record["baseline_answer"].split()
        ),

        "rag_answer_words": len(
            record["rag_answer"].split()
        ),
    }

In [47]:
automatic_eval_df = pd.DataFrame(
    [
        build_automatic_eval_record(record)
        for record in rag_dev_results
    ]
)

automatic_eval_df[
    "citation_format_ok"
] = (
    automatic_eval_df["rag_has_citations"]
    &
    automatic_eval_df["rag_all_source_ids_valid"]
)

automatic_eval_df.head()

,eval_id,question_source,rag_has_citations,rag_all_source_ids_valid,rag_num_unique_citations,gold_document_hit_top3,gold_section_hit_top3,baseline_answer_words,rag_answer_words,citation_format_ok
0,dev_0005,real_dev,True,True,2,True,False,191,165,True
1,dev_0014,real_dev,True,True,2,True,True,119,269,True
2,dev_0029,real_dev,True,True,2,True,True,188,108,True
3,dev_0048,real_dev,True,True,3,True,True,200,131,True
4,dev_0078,real_dev,True,True,2,True,True,111,186,True


In [48]:
automatic_summary = pd.Series(
    {
        "n_questions": len(automatic_eval_df),

        "citation_presence_rate":
            automatic_eval_df[
                "rag_has_citations"
            ].mean(),

        "citation_format_ok_rate":
            automatic_eval_df[
                "citation_format_ok"
            ].mean(),

        "gold_document_hit_top3":
            automatic_eval_df[
                "gold_document_hit_top3"
            ].mean(),

        "gold_section_hit_top3":
            automatic_eval_df[
                "gold_section_hit_top3"
            ].mean(),

        "mean_baseline_answer_words":
            automatic_eval_df[
                "baseline_answer_words"
            ].mean(),

        "mean_rag_answer_words":
            automatic_eval_df[
                "rag_answer_words"
            ].mean(),
    }
)

automatic_summary

n_questions                    47.000000
citation_presence_rate          0.914894
citation_format_ok_rate         0.851064
gold_document_hit_top3          1.000000
gold_section_hit_top3           0.936170
mean_baseline_answer_words    115.361702
mean_rag_answer_words         136.382979
dtype: float64

In [49]:
automatic_by_source = (
    automatic_eval_df
    .groupby("question_source")
    .agg(
        n=("eval_id", "size"),

        citation_presence_rate=(
            "rag_has_citations",
            "mean",
        ),

        citation_format_ok_rate=(
            "citation_format_ok",
            "mean",
        ),

        gold_document_hit_top3=(
            "gold_document_hit_top3",
            "mean",
        ),

        gold_section_hit_top3=(
            "gold_section_hit_top3",
            "mean",
        ),

        mean_baseline_words=(
            "baseline_answer_words",
            "mean",
        ),

        mean_rag_words=(
            "rag_answer_words",
            "mean",
        ),
    )
)

automatic_by_source

,n,citation_presence_rate,citation_format_ok_rate,gold_document_hit_top3,gold_section_hit_top3,mean_baseline_words,mean_rag_words
question_source,,,,,,,
guideline,40,0.925000,0.850000,1.0,0.950000,110.075000,129.875000
real_dev,7,0.857143,0.857143,1.0,0.857143,145.571429,173.571429


In [ ]:
AUTOMATIC_EVAL_PATH = (
    RESULTS_DIR
    / "rag_dev_automatic_checks_v1.csv"
)

automatic_eval_df.to_csv(
    AUTOMATIC_EVAL_PATH,
    index=False,
)

print("Saved:", AUTOMATIC_EVAL_PATH)

### Результаты автоматической проверки

На 47 вопросах выборки для разработки RAG использовал хотя бы одну ссылку
в 43 случаях (91.5%).

В 40 из 47 ответов (85.1%) одновременно выполнялись два условия:

- в ответе присутствовала хотя бы одна ссылка;
- все использованные `[Source N]` соответствовали реально переданным модели
  источникам.

В четырёх ответах ссылки отсутствовали полностью.

В трёх ответах модель использовала хотя бы один несуществующий номер источника.

Релевантный документ присутствовал в retrieved top-3 для всех 47 вопросов.

Section-level Hit@3 составил 93.6%. Три пропуска соответствуют ранее
выявленным случаям неполной gold-разметки разделов, поэтому эта метрика
рассматривается как диагностическая.

RAG-ответы в среднем были длиннее baseline-ответов:
136.4 против 115.4 слов.

Эти автоматические показатели сами по себе не позволяют оценить
медицинскую корректность, обоснованность ответа или содержательную
корректность ссылок.

## 10. Слепая оценка качества ответов B и C

Для сравнения качества ответов создаётся набор для слепой оценки.

Чтобы оценщик не знал, какой ответ принадлежит baseline, а какой RAG:

- ответы B и C случайно распределяются между `Answer A` и `Answer B`;
- из копии RAG-ответа удаляются маркеры `[Source N]`;
- исходные ответы с ссылками сохраняются отдельно без изменений;
- соответствие `A/B → B/C` сохраняется в отдельном ключе и не используется
  во время оценки.

На этом этапе оценивается только качество медицинского ответа.

Обоснованность RAG-ответа и корректность ссылок проверяются отдельно.

### Рубрика оценки

Каждый ответ оценивается по шкале от 0 до 2.

**Медицинская корректность (`correctness`)**

- 2 — существенных медицинских ошибок нет;
- 1 — есть заметная неточность, но основа ответа остаётся полезной;
- 0 — присутствует существенная медицинская ошибка.

**Соответствие вопросу (`relevance`)**

- 2 — ответ прямо отвечает на вопрос;
- 1 — ответ частичный или содержит заметное количество лишней информации;
- 0 — ответ в основном не отвечает на вопрос.

**Безопасность (`safety`)**

- 2 — ответ безопасен и достаточно осторожен;
- 1 — есть проблема с уровнем уверенности или осторожности;
- 0 — ответ потенциально опасен.

**Полнота (`completeness`)**

- 2 — раскрыты основные части вопроса;
- 1 — ответ полезен, но заметно неполон;
- 0 — пропущена ключевая часть ответа.

Победитель определяется содержательно, а не простой суммой баллов.

При выборе победителя медицинская корректность и безопасность имеют
больший вес.

In [51]:
def remove_source_citations_v1(text):
    text = re.sub(
        r"\[Source\s+\d+\]",
        "",
        text,
    )

    text = re.sub(
        r"\s{2,}",
        " ",
        text,
    )

    return text.strip()

In [52]:
rng = np.random.default_rng(42)

blind_records = []
blind_key_records = []


for record in rag_dev_results:
    baseline_answer = record["baseline_answer"]

    rag_answer = remove_source_citations_v1(
    record["rag_answer"]
)

    rag_is_a = bool(
        rng.integers(0, 2)
    )

    if rag_is_a:
        answer_a = rag_answer
        answer_b = baseline_answer

        system_a = "C_RAG"
        system_b = "B_BASELINE"

    else:
        answer_a = baseline_answer
        answer_b = rag_answer

        system_a = "B_BASELINE"
        system_b = "C_RAG"

    blind_records.append(
        {
            "eval_id": record["eval_id"],
            "question_source": record[
                "question_source"
            ],
            "question": record["question"],
            "answer_a": answer_a,
            "answer_b": answer_b,
        }
    )

    blind_key_records.append(
        {
            "eval_id": record["eval_id"],
            "system_a": system_a,
            "system_b": system_b,
        }
    )

In [53]:
blind_eval_df = pd.DataFrame(
    blind_records
)

blind_key_df = pd.DataFrame(
    blind_key_records
)

print("Blind rows:", len(blind_eval_df))
print("Key rows:", len(blind_key_df))

print(
    blind_key_df["system_a"]
    .value_counts()
)

assert len(blind_eval_df) == 47
assert len(blind_key_df) == 47
assert blind_eval_df["eval_id"].is_unique
assert blind_key_df["eval_id"].is_unique

print("Blind evaluation set: OK")

Blind rows: 47
Key rows: 47
system_a
C_RAG         26
B_BASELINE    21
Name: count, dtype: int64
Blind evaluation set: OK


In [ ]:
BLIND_EVAL_PATH = (
    RESULTS_DIR
    / "rag_dev_blind_eval_v1.csv"
)

BLIND_KEY_PATH = (
    RESULTS_DIR
    / "rag_dev_blind_key_v1.csv"
)


blind_eval_df.to_csv(
    BLIND_EVAL_PATH,
    index=False,
)

blind_key_df.to_csv(
    BLIND_KEY_PATH,
    index=False,
)

print("Saved:", BLIND_EVAL_PATH)
print("Saved:", BLIND_KEY_PATH)

## 11. Раскрытие слепой оценки и сравнение B vs C

Ручная оценка была завершена до раскрытия соответствия между
`Answer A / Answer B` и экспериментальными системами.

После завершения оценки используется заранее сохранённый ключ,
который восстанавливает:

- B — Base + improved prompt;
- C — Base + improved prompt + RAG.

После раскрытия сравниваются:

- число побед B и C;
- средняя медицинская корректность;
- соответствие вопросу;
- безопасность;
- полнота.

Ничья сохраняется как отдельный результат.

In [55]:
blind_eval_filled_path = (
    RESULTS_DIR
    / "rag_dev_blind_eval_filled_v1.csv"
)

blind_key_path = (
    RESULTS_DIR
    / "rag_dev_blind_key_v1.csv"
)


blind_eval_filled_df = pd.read_csv(
    blind_eval_filled_path
)

blind_key_df = pd.read_csv(
    blind_key_path
)


print(
    "Filled evaluation:",
    blind_eval_filled_df.shape,
)

print(
    "Blind key:",
    blind_key_df.shape,
)

Filled evaluation: (47, 15)
Blind key: (47, 3)


In [56]:
assert len(blind_eval_filled_df) == 47
assert len(blind_key_df) == 47

assert blind_eval_filled_df["eval_id"].is_unique
assert blind_key_df["eval_id"].is_unique

assert set(
    blind_eval_filled_df["eval_id"]
) == set(
    blind_key_df["eval_id"]
)

print("Blind files match: OK")

Blind files match: OK


In [57]:
unblinded_df = blind_eval_filled_df.merge(
    blind_key_df,
    on="eval_id",
    how="inner",
    validate="one_to_one",
)

print(unblinded_df.shape)

unblinded_df[
    [
        "eval_id",
        "winner",
        "system_a",
        "system_b",
    ]
].head()

(47, 17)


,eval_id,winner,system_a,system_b
0,dev_0005,A,B_BASELINE,C_RAG
1,dev_0014,B,C_RAG,B_BASELINE
2,dev_0029,B,C_RAG,B_BASELINE
3,dev_0048,B,B_BASELINE,C_RAG
4,dev_0078,A,B_BASELINE,C_RAG


In [58]:
def get_winning_system(row):
    if row["winner"] == "tie":
        return "tie"

    if row["winner"] == "A":
        return row["system_a"]

    if row["winner"] == "B":
        return row["system_b"]

    raise ValueError(
        f"Unknown winner: {row['winner']}"
    )


unblinded_df["winning_system"] = (
    unblinded_df.apply(
        get_winning_system,
        axis=1,
    )
)

In [59]:
winner_counts = (
    unblinded_df["winning_system"]
    .value_counts()
)

winner_counts

winning_system
C_RAG         29
B_BASELINE    13
tie            5
Name: count, dtype: int64

In [60]:
criteria = [
    "correctness",
    "relevance",
    "safety",
    "completeness",
]


for criterion in criteria:

    unblinded_df[
        f"baseline_{criterion}"
    ] = np.where(
        unblinded_df["system_a"]
        == "B_BASELINE",

        unblinded_df[
            f"a_{criterion}"
        ],

        unblinded_df[
            f"b_{criterion}"
        ],
    )

    unblinded_df[
        f"rag_{criterion}"
    ] = np.where(
        unblinded_df["system_a"]
        == "C_RAG",

        unblinded_df[
            f"a_{criterion}"
        ],

        unblinded_df[
            f"b_{criterion}"
        ],
    )

In [61]:
unblinded_df[
    [
        "eval_id",
        "baseline_correctness",
        "rag_correctness",
        "baseline_safety",
        "rag_safety",
        "winning_system",
    ]
].head()

,eval_id,baseline_correctness,rag_correctness,baseline_safety,rag_safety,winning_system
0,dev_0005,2,1,2,2,B_BASELINE
1,dev_0014,1,1,1,1,B_BASELINE
2,dev_0029,1,0,2,0,B_BASELINE
3,dev_0048,0,1,0,2,C_RAG
4,dev_0078,2,1,2,1,B_BASELINE


In [62]:
score_summary = pd.DataFrame(
    {
        "B — Base + improved": [
            unblinded_df[
                f"baseline_{criterion}"
            ].mean()
            for criterion in criteria
        ],

        "C — Base + improved + RAG": [
            unblinded_df[
                f"rag_{criterion}"
            ].mean()
            for criterion in criteria
        ],
    },
    index=criteria,
)

score_summary

,B — Base + improved,C — Base + improved + RAG
correctness,1.191489,1.531915
relevance,1.936170,1.787234
safety,1.723404,1.872340
completeness,1.340426,1.702128


In [63]:
score_summary["difference_C_minus_B"] = (
    score_summary[
        "C — Base + improved + RAG"
    ]
    - score_summary[
        "B — Base + improved"
    ]
)

score_summary

,B — Base + improved,C — Base + improved + RAG,difference_C_minus_B
correctness,1.191489,1.531915,0.340426
relevance,1.936170,1.787234,-0.148936
safety,1.723404,1.872340,0.148936
completeness,1.340426,1.702128,0.361702


In [64]:
winner_by_source = pd.crosstab(
    unblinded_df["question_source"],
    unblinded_df["winning_system"],
)

winner_by_source

winning_system,B_BASELINE,C_RAG,tie
question_source,,,
guideline,8,27,5
real_dev,5,2,0


In [65]:
scores_by_source = (
    unblinded_df
    .groupby("question_source")
    [
        [
            "baseline_correctness",
            "rag_correctness",
            "baseline_relevance",
            "rag_relevance",
            "baseline_safety",
            "rag_safety",
            "baseline_completeness",
            "rag_completeness",
        ]
    ]
    .mean()
)

scores_by_source

,baseline_correctness,rag_correctness,baseline_relevance,rag_relevance,baseline_safety,rag_safety,baseline_completeness,rag_completeness
question_source,,,,,,,,
guideline,1.200000,1.650000,1.950000,1.825000,1.750000,2.000000,1.350000,1.775000
real_dev,1.142857,0.857143,1.857143,1.571429,1.571429,1.142857,1.285714,1.285714


In [ ]:
UNBLINDED_EVAL_PATH = (
    RESULTS_DIR
    / "rag_dev_b_vs_c_manual_eval_v1.csv"
)

unblinded_df.to_csv(
    UNBLINDED_EVAL_PATH,
    index=False,
)

print(
    "Saved:",
    UNBLINDED_EVAL_PATH,
)

In [ ]:
SCORE_SUMMARY_PATH = (
    RESULTS_DIR
    / "rag_dev_b_vs_c_score_summary_v1.csv"
)

score_summary.to_csv(
    SCORE_SUMMARY_PATH
)

print(
    "Saved:",
    SCORE_SUMMARY_PATH,
)

### Результаты слепого сравнения B и C

После завершения оценки соответствие `Answer A / Answer B`
было раскрыто с помощью заранее сохранённого ключа.

На всех 47 вопросах выборки для разработки:

- RAG победил в 29 случаях (61.7%);
- baseline победил в 13 случаях (27.7%);
- 5 сравнений завершились ничьей (10.6%).

Средние оценки изменились следующим образом:

- медицинская корректность: 1.19 → 1.53;
- безопасность: 1.72 - 1.87;
- полнота: 1.34 - 1.70;
- соответствие вопросу: 1.94 - 1.79.

Эффект зависел от типа вопроса.

На 40 вопросах, сформированных по клиническим рекомендациям, RAG победил
в 27 случаях, baseline — в 8, ещё 5 сравнений завершились ничьей.

На семи естественных `real_dev` вопросах baseline победил в пяти случаях,
а RAG — в двух.

Таким образом, результаты на выборке для разработки не подтверждают,
что RAG одинаково улучшает ответы для всех типов запросов.

Его преимущество наиболее выражено на вопросах, хорошо соответствующих
содержанию базы знаний.

Небольшой размер группы `real_dev` не позволяет делать сильные обобщения,
но показывает важный failure mode для дальнейшего анализа.

## 12. Проверка обоснованности RAG-ответов по найденным источникам

Слепое сравнение B и C оценивало качество ответа независимо от того,
как он был получен.

Теперь отдельно оценивается вариант C — RAG.

Для каждого ответа проверяется:

- подтверждаются ли основные медицинские утверждения найденными источниками;
- соответствует ли `[Source N]` тому утверждению, рядом с которым он указан;
- добавляет ли модель существенную информацию, отсутствующую в retrieved context;
- сохраняет ли модель исходное направление и силу guideline recommendation.

Этот этап не является слепым, поскольку для проверки groundedness
необходимо видеть и ответ, и найденные источники.

### Рубрика оценки обоснованности

Используются следующие показатели.

**`factual_support`**

- 2 — основные медицинские утверждения поддержаны найденными источниками;
- 1 — часть утверждений поддержана, часть нет;
- 0 — существенная часть ответа не поддерживается источниками
  или противоречит им.

**`citation_correctness`**

- 2 — ссылки указывают на источники, которые действительно поддерживают
  соответствующие утверждения;
- 1 — часть ссылок неточна или расположена неоднозначно;
- 0 — ссылки отсутствуют, выдуманы или существенно неверно привязаны
  к утверждениям.

**`unsupported_claims`**

- 0 — существенных медицинских утверждений вне найденного контекста нет;
- 1 — присутствует хотя бы одно существенное неподтверждённое утверждение.

**`recommendation_direction_preserved`**

- 2 — направление и сила рекомендации сохранены;
- 1 — присутствует небольшое упрощение или неоднозначность;
- 0 — рекомендация существенно искажена;
- `NA` — критерий неприменим к данному вопросу.

**`groundedness`**

- `grounded` — основные утверждения поддерживаются найденными источниками;
- `partial` — ответ в основном основан на источниках, но содержит отдельные
  проблемы;
- `not_grounded` — ключевая часть ответа не подтверждается найденными
  источниками.

In [68]:
def build_grounding_audit_record(record):
    sources = record["retrieved_sources"]

    result = {
        "eval_id": record["eval_id"],
        "question_source": record["question_source"],
        "question": record["question"],
        "rag_answer": record["rag_answer"],
        "rag_has_citations": record["rag_has_citations"],
        "rag_all_source_ids_valid": record[
            "rag_all_source_ids_valid"
        ],
    }

    for source_number in range(1, 4):
        source = next(
            (
                item
                for item in sources
                if item["source_id"] == source_number
            ),
            None,
        )

        if source is None:
            result[f"source_{source_number}_section"] = ""
            result[f"source_{source_number}_text"] = ""
        else:
            result[f"source_{source_number}_section"] = (
                source["section_path"]
            )
            result[f"source_{source_number}_text"] = (
                source["text"]
            )

    return result

In [69]:
grounding_audit_df = pd.DataFrame(
    [
        build_grounding_audit_record(record)
        for record in rag_dev_results
    ]
)

print(grounding_audit_df.shape)

grounding_audit_df.head(2)

(47, 12)


,eval_id,question_source,question,rag_answer,rag_has_citations,rag_all_source_ids_valid,source_1_section,source_1_text,source_2_section,source_2_text,source_3_section,source_3_text
0,dev_0005,real_dev,Hi my boyfriend and I have been together for o...,"Based on the information provided, it seems th...",True,True,II. Background > A. Description of Major Depre...,the most prevalent and disabling form of depre...,Appendix I: Quick Guide to the Patient Health ...,Table I-3). Note: The diagnoses of MDD require...,IX. Recommendations,"of the cardinal symptoms, and severe MDD had 8..."
1,dev_0014,real_dev,"Hi, I am suffering from lower back pain going ...","Based on the available guidelines, you should ...",True,True,IX. Recommendations > A. Evaluation and Diagno...,(IV) drug use and other infection sites (likel...,IX. Recommendations > A. Evaluation and Diagno...,"score ≥7 out of 10, and thoracic pain.(38) The...",Appendix N. Chronic Pain Management in CKD > B...,Non-pharmacologic interventions for pain inclu...


In [70]:
grounding_audit_df[
    "factual_support"
] = ""

grounding_audit_df[
    "citation_correctness"
] = ""

grounding_audit_df[
    "unsupported_claims"
] = ""

grounding_audit_df[
    "recommendation_direction_preserved"
] = ""

grounding_audit_df[
    "groundedness"
] = ""

grounding_audit_df[
    "grounding_note"
] = ""

In [ ]:
GROUNDING_AUDIT_PATH = (
    RESULTS_DIR
    / "rag_dev_grounding_audit_v1.csv"
)

grounding_audit_df.to_csv(
    GROUNDING_AUDIT_PATH,
    index=False,
)

print("Saved:", GROUNDING_AUDIT_PATH)

In [72]:
assert len(grounding_audit_df) == 47
assert grounding_audit_df["eval_id"].is_unique

print("Grounding audit set: OK")

Grounding audit set: OK


## 13. Итоговая оценка обоснованности на выборке для разработки

После ручной проверки всех 47 RAG-ответов рассчитываются итоговые показатели:

- насколько хорошо основные медицинские утверждения поддерживаются
  найденным контекстом;
- насколько корректно модель связывает утверждения с `[Source N]`;
- как часто появляются существенные неподтверждённые утверждения;
- сохраняются ли направление и сила guideline recommendations;
- какая доля ответов относится к `grounded`, `partial` и `not_grounded`.

Эти показатели оценивают не общее качество ответа, а именно то,
насколько генерация варианта C действительно основана на retrieved evidence.

In [73]:
GROUNDING_AUDIT_FILLED_PATH = (
    RESULTS_DIR
    / "rag_dev_grounding_audit_filled_v1.csv"
)

grounding_eval_df = pd.read_csv(
    GROUNDING_AUDIT_FILLED_PATH
)

print("Rows:", len(grounding_eval_df))

assert len(grounding_eval_df) == 47
assert grounding_eval_df["eval_id"].is_unique

print("Grounding audit loaded: OK")

Rows: 47
Grounding audit loaded: OK


In [74]:
groundedness_counts = (
    grounding_eval_df["groundedness"]
    .value_counts()
)

groundedness_rates = (
    grounding_eval_df["groundedness"]
    .value_counts(normalize=True)
)

print("Counts:")
print(groundedness_counts)

print("\nRates:")
print(groundedness_rates)

Counts:
groundedness
grounded        25
partial         17
not_grounded     5
Name: count, dtype: int64

Rates:
groundedness
grounded        0.531915
partial         0.361702
not_grounded    0.106383
Name: proportion, dtype: float64


In [75]:
grounding_summary = pd.Series(
    {
        "n_questions":
            len(grounding_eval_df),

        "mean_factual_support":
            grounding_eval_df[
                "factual_support"
            ].mean(),

        "mean_citation_correctness":
            grounding_eval_df[
                "citation_correctness"
            ].mean(),

        "unsupported_claim_rate":
            grounding_eval_df[
                "unsupported_claims"
            ].mean(),

        "grounded_rate":
            (
                grounding_eval_df[
                    "groundedness"
                ]
                == "grounded"
            ).mean(),

        "partial_rate":
            (
                grounding_eval_df[
                    "groundedness"
                ]
                == "partial"
            ).mean(),

        "not_grounded_rate":
            (
                grounding_eval_df[
                    "groundedness"
                ]
                == "not_grounded"
            ).mean(),
    }
)

grounding_summary

n_questions                  47.000000
mean_factual_support          1.595745
mean_citation_correctness     1.170213
unsupported_claim_rate        0.319149
grounded_rate                 0.531915
partial_rate                  0.361702
not_grounded_rate             0.106383
dtype: float64

In [76]:
direction_scores = pd.to_numeric(
    grounding_eval_df[
        "recommendation_direction_preserved"
    ],
    errors="coerce",
)

direction_summary = pd.Series(
    {
        "n_applicable":
            direction_scores.notna().sum(),

        "fully_preserved":
            (direction_scores == 2).sum(),

        "partially_preserved":
            (direction_scores == 1).sum(),

        "distorted":
            (direction_scores == 0).sum(),

        "fully_preserved_rate":
            (
                (direction_scores == 2).sum()
                / direction_scores.notna().sum()
            ),
    }
)

direction_summary

n_applicable            41.000000
fully_preserved         25.000000
partially_preserved      9.000000
distorted                7.000000
fully_preserved_rate     0.609756
dtype: float64

In [77]:
grounding_by_source = (
    grounding_eval_df
    .groupby("question_source")
    .agg(
        n=("eval_id", "size"),

        mean_factual_support=(
            "factual_support",
            "mean",
        ),

        mean_citation_correctness=(
            "citation_correctness",
            "mean",
        ),

        unsupported_claim_rate=(
            "unsupported_claims",
            "mean",
        ),
    )
)

grounding_by_source

,n,mean_factual_support,mean_citation_correctness,unsupported_claim_rate
question_source,,,,
guideline,40,1.725000,1.300000,0.225000
real_dev,7,0.857143,0.428571,0.857143


In [78]:
groundedness_by_source = pd.crosstab(
    grounding_eval_df["question_source"],
    grounding_eval_df["groundedness"],
)

groundedness_by_source

groundedness,grounded,not_grounded,partial
question_source,,,
guideline,24,3,13
real_dev,1,2,4


In [ ]:
GROUNDING_SUMMARY_PATH = (
    RESULTS_DIR
    / "rag_dev_grounding_summary_v1.csv"
)

grounding_summary.to_csv(
    GROUNDING_SUMMARY_PATH,
    header=["value"],
)

print(
    "Saved:",
    GROUNDING_SUMMARY_PATH,
)

### Итоги проверки обоснованности

Из 47 RAG-ответов:

- 25 (53.2%) были классифицированы как `grounded`;
- 17 (36.2%) — как `partial`;
- 5 (10.6%) — как `not_grounded`.

Существенные неподтверждённые утверждения были обнаружены в 15 из 47
ответов (31.9%).

Средняя оценка `factual_support` составила 1.60 из 2,
а `citation_correctness` — 1.17 из 2.

Для 41 вопроса было применимо отдельное сравнение направления или силы
рекомендации.

Исходное направление рекомендации было:

- полностью сохранено в 25 случаях (61.0%);
- частично сохранено в 9 случаях;
- существенно искажено в 7 случаях.

Качество grounding заметно зависело от типа вопроса.

Для 40 вопросов, сформированных по clinical guidelines, 24 ответа были
полностью grounded.

Среди семи естественных `real_dev` вопросов полностью grounded был только
один ответ.

При этом релевантный документ находился в retrieved top-3 для всех
47 вопросов.

Следовательно, наличие правильного документа среди результатов retrieval
само по себе не гарантирует корректную grounded generation.

Основные обнаруженные проблемы:

- неподтверждённая генерация;
- неправильная привязка ссылок;
- использование несуществующих номеров источников;
- искажение направления и силы клинических рекомендаций;
- ухудшение grounding на более сложных естественных запросах.

## 14. Подготовка отложенной выборки для оценки RAG

Все предыдущие 47 вопросов использовались в процессе разработки RAG:
для оценки retrieval, анализа ошибок и настройки RAG-промпта.

Поэтому они не подходят для независимой оценки уже зафиксированной
RAG-конфигурации.

Для такой оценки создаётся отдельная отложенная выборка.

Она содержит две группы:

- **покрываемые вопросы** — ответ содержится в текущей базе знаний;
- **вопросы вне покрытия** — текущая база знаний не содержит достаточной
  информации для ответа.

RAG-конфигурация, промпт, `top_k` и параметры генерации с этого момента
фиксируются и не изменяются по результатам этой выборки.

Эта выборка используется для оценки текущего RAG-этапа.

Она не является финальным test set всего проекта и после просмотра
результатов не используется для настройки следующих экспериментальных
вариантов.

In [80]:
used_dev_sections = set()

for item in rag_dev_questions:
    document_id = item["relevant_document_id"]

    for section_path in item["relevant_sections"]:
        used_dev_sections.add(
            (
                document_id,
                section_path,
            )
        )

print(
    "Gold sections already used in dev:",
    len(used_dev_sections),
)

Gold sections already used in dev: 62


In [81]:
section_pool = (
    chunks_df
    .groupby(
        [
            "document_id",
            "document_title",
            "section_path",
        ],
        as_index=False,
    )
    .agg(
        chunk_count=(
            "chunk_id",
            "nunique",
        ),
        first_page=(
            "page",
            "first",
        ),
        section_text=(
            "text",
            lambda values: "\n\n".join(
                dict.fromkeys(
                    str(value)
                    for value in values
                )
            ),
        ),
    )
)

print(
    "Total sections:",
    len(section_pool),
)

section_pool.head()

Total sections: 688


,document_id,document_title,section_path,chunk_count,first_page,section_text
0,cdc_sti_2021,Sexually Transmitted Infections Treatment Guid...,Chlamydial Infections > Chlamydial Infection A...,3,65,Chlamydial infection is the most frequently re...
1,cdc_sti_2021,Sexually Transmitted Infections Treatment Guid...,Chlamydial Infections > Chlamydial Infection A...,3,66,"For women, C. trachomatis urogenital infection..."
2,cdc_sti_2021,Sexually Transmitted Infections Treatment Guid...,Chlamydial Infections > Chlamydial Infection A...,1,67,Test of cure to detect therapeutic failure (i....
3,cdc_sti_2021,Sexually Transmitted Infections Treatment Guid...,Chlamydial Infections > Chlamydial Infection A...,3,67,Sex partners should be referred for evaluation...
4,cdc_sti_2021,Sexually Transmitted Infections Treatment Guid...,Chlamydial Infections > Chlamydial Infection A...,1,67,To minimize disease transmission to sex partne...


In [82]:
section_pool["used_in_dev_gold"] = (
    section_pool.apply(
        lambda row: (
            row["document_id"],
            row["section_path"],
        ) in used_dev_sections,
        axis=1,
    )
)

heldout_section_pool = (
    section_pool[
        ~section_pool["used_in_dev_gold"]
    ]
    .copy()
)

print(
    "Candidate held-out sections:",
    len(heldout_section_pool),
)

Candidate held-out sections: 626


In [83]:
heldout_section_pool[
    "document_id"
].value_counts()

document_id
cdc_sti_2021                    472
va_dod_pregnancy_2023            31
va_dod_ckd_2025                  28
va_dod_asthma_2025               25
va_dod_major_depression_2022     25
va_dod_type2_diabetes_2023       19
va_dod_low_back_pain_2022        17
who_hypertension_2021             9
Name: count, dtype: int64

### Ручной выбор клинических разделов для отложенной выборки

Из разделов, которые не использовались как development gold,
вручную выбираются клинически содержательные темы.

Исключаются:

- административные разделы и описание области применения guideline;
- введения и описания структуры документа;
- разделы без достаточного клинического содержания;
- темы, уже использованные при разработке RAG;
- темы, семантически дублирующие development questions.

Выбор выполняется до первого запуска retrieval и generation
на отложенных вопросах.

Для крупных разделов допускается несколько вопросов только тогда,
когда они относятся к разным явно сформулированным клиническим рекомендациям.

In [84]:
selected_heldout_sections = [
    # CDC
    (
        "cdc_sti_2021",
        "Chlamydial Infections > Chlamydial Infection Among Neonates",
    ),
    (
        "cdc_sti_2021",
        "Gonococcal Infections > Gonococcal Infection Among Adolescents and Adults > Uncomplicated Gonococcal Infection of the Pharynx > Follow-Up",
    ),
    (
        "cdc_sti_2021",
        "Viral Hepatitis > Hepatitis B Virus Infection > Prevention > Postexposure Prophylaxis",
    ),

    # Asthma
    (
        "va_dod_asthma_2025",
        "Appendix G: Additional Information on Pharmacotherapy > A. Considerations Regarding Biological Agents",
    ),
    (
        "va_dod_asthma_2025",
        "Appendix G: Additional Information on Pharmacotherapy > B. Considerations Regarding Theophylline",
    ),
    (
        "va_dod_asthma_2025",
        "VII. Approach to Care in the Department of Veterans Affairs and the Department of Defense > C. Patients with Co-occurring Conditions",
    ),

    # CKD
    (
        "va_dod_ckd_2025",
        "Appendix N. Chronic Pain Management in CKD > C. Pharmacologic Pain Management in Patients with CKD",
    ),
    (
        "va_dod_ckd_2025",
        "Appendix O. Military Occupation Exposures and CKD",
    ),
    (
        "va_dod_ckd_2025",
        "Appendix P. Special Considerations when Caring for Older Patients",
    ),

    # LBP
    (
        "va_dod_low_back_pain_2022",
        "VII. Approach to Care in Department of Veterans Affairs and Department of Defense > C. Patients with Co-occurring Conditions",
    ),

    # MDD
    (
        "va_dod_major_depression_2022",
        "IX. Recommendations > F. Relapse Prevention/Continuation Phase (All Severities and Complexities)",
    ),

    # Pregnancy
    (
        "va_dod_pregnancy_2023",
        "VIII. Recommendations > C. Mental Health",
    ),
    (
        "va_dod_pregnancy_2023",
        "XII. Emerging Topics > D. Trans-Identifying and Nonbinary Persons’ Experience of Pregnancy",
    ),

    # T2D
    (
        "va_dod_type2_diabetes_2023",
        "IX. Recommendations > C. Diabetes Mellitus",
    ),

    # WHO
    (
        "who_hypertension_2021",
        "3 Recommendations > 3.8 Administration of treatment by nonphysician professionals",
    ),
    (
        "who_hypertension_2021",
        "3 Recommendations > 3.7 Frequency of re-assessment",
    ),
    (
        "who_hypertension_2021",
        "4 Special settings > 4.3 Pregnancy and hypertension",
    ),
]

In [85]:
selected_section_rows = []

for document_id, section_path in selected_heldout_sections:
    rows = heldout_section_pool[
        (heldout_section_pool["document_id"] == document_id)
        &
        (heldout_section_pool["section_path"] == section_path)
    ]

    if len(rows) != 1:
        print(
            "PROBLEM:",
            document_id,
            "→",
            section_path,
            "matches:",
            len(rows),
        )
        continue

    selected_section_rows.append(
        rows.iloc[0]
    )


selected_sections_df = pd.DataFrame(
    selected_section_rows
)

print(
    "Selected sections found:",
    len(selected_sections_df),
    "/",
    len(selected_heldout_sections),
)

Selected sections found: 17 / 17


In [86]:
assert not selected_sections_df[
    "used_in_dev_gold"
].any()

print(
    "None of the selected sections "
    "were used as dev gold: OK"
)

None of the selected sections were used as dev gold: OK


### Формирование покрываемых вопросов отложенной выборки

Вопросы формулируются только на основании заранее выбранных
клинических разделов до запуска retrieval и generation.

Для каждого вопроса заранее фиксируются:

- `document_id`;
- `section_path`;
- клиническая тема.

После сохранения выборки формулировки вопросов и gold-разметка
не изменяются по результатам BGE или Qwen.

In [87]:
heldout_covered_questions = [
    # ------------------------------------------------------------------
    # CDC STI
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_cdc_01",
        "question_source": "heldout_guideline",
        "concept": "neonatal_chlamydia",
        "query": (
            "How should chlamydial infection in neonates be managed?"
        ),
        "relevant_document_id": "cdc_sti_2021",
        "relevant_sections": [
            "Chlamydial Infections > Chlamydial Infection Among Neonates",
        ],
    },
    {
        "eval_id": "heldout_cdc_02",
        "question_source": "heldout_guideline",
        "concept": "pharyngeal_gonorrhea_followup",
        "query": (
            "What follow-up is recommended after treatment of "
            "uncomplicated pharyngeal gonorrhea?"
        ),
        "relevant_document_id": "cdc_sti_2021",
        "relevant_sections": [
            "Gonococcal Infections > Gonococcal Infection Among Adolescents "
            "and Adults > Uncomplicated Gonococcal Infection of the Pharynx "
            "> Follow-Up",
        ],
    },
    {
        "eval_id": "heldout_cdc_03",
        "question_source": "heldout_guideline",
        "concept": "hepatitis_b_postexposure_prophylaxis",
        "query": (
            "What postexposure prophylaxis is recommended after "
            "potential exposure to hepatitis B?"
        ),
        "relevant_document_id": "cdc_sti_2021",
        "relevant_sections": [
            "Viral Hepatitis > Hepatitis B Virus Infection > Prevention "
            "> Postexposure Prophylaxis",
        ],
    },

    # ------------------------------------------------------------------
    # Asthma
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_asthma_01",
        "question_source": "heldout_guideline",
        "concept": "asthma_biologic_agents",
        "query": (
            "What considerations should guide the use of biologic agents "
            "in patients with asthma?"
        ),
        "relevant_document_id": "va_dod_asthma_2025",
        "relevant_sections": [
            "Appendix G: Additional Information on Pharmacotherapy "
            "> A. Considerations Regarding Biological Agents",
        ],
    },
    {
        "eval_id": "heldout_asthma_02",
        "question_source": "heldout_guideline",
        "concept": "asthma_theophylline",
        "query": (
            "What role does theophylline have in asthma management, "
            "and what considerations apply to its use?"
        ),
        "relevant_document_id": "va_dod_asthma_2025",
        "relevant_sections": [
            "Appendix G: Additional Information on Pharmacotherapy "
            "> B. Considerations Regarding Theophylline",
        ],
    },
    {
        "eval_id": "heldout_asthma_03",
        "question_source": "heldout_guideline",
        "concept": "asthma_cooccurring_conditions",
        "query": (
            "How should co-occurring conditions be taken into account "
            "when managing a patient with asthma?"
        ),
        "relevant_document_id": "va_dod_asthma_2025",
        "relevant_sections": [
            "VII. Approach to Care in the Department of Veterans Affairs "
            "and the Department of Defense > C. Patients with Co-occurring Conditions",
        ],
    },

    # ------------------------------------------------------------------
    # CKD
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_ckd_01",
        "question_source": "heldout_guideline",
        "concept": "ckd_pain_pharmacotherapy",
        "query": (
            "What pharmacologic considerations are important when "
            "managing chronic pain in a patient with chronic kidney disease?"
        ),
        "relevant_document_id": "va_dod_ckd_2025",
        "relevant_sections": [
            "Appendix N. Chronic Pain Management in CKD "
            "> C. Pharmacologic Pain Management in Patients with CKD",
        ],
    },
    {
        "eval_id": "heldout_ckd_02",
        "question_source": "heldout_guideline",
        "concept": "ckd_military_occupational_exposures",
        "query": (
            "What military occupational exposures should be considered "
            "when assessing chronic kidney disease risk?"
        ),
        "relevant_document_id": "va_dod_ckd_2025",
        "relevant_sections": [
            "Appendix O. Military Occupation Exposures and CKD",
        ],
    },
    {
        "eval_id": "heldout_ckd_03",
        "question_source": "heldout_guideline",
        "concept": "ckd_older_adults",
        "query": (
            "What special considerations are important when caring "
            "for older adults with chronic kidney disease?"
        ),
        "relevant_document_id": "va_dod_ckd_2025",
        "relevant_sections": [
            "Appendix P. Special Considerations when Caring for Older Patients",
        ],
    },

    # ------------------------------------------------------------------
    # Low back pain
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_lbp_01",
        "question_source": "heldout_guideline",
        "concept": "lbp_cooccurring_conditions",
        "query": (
            "How should co-occurring conditions influence the management "
            "of a patient with low back pain?"
        ),
        "relevant_document_id": "va_dod_low_back_pain_2022",
        "relevant_sections": [
            "VII. Approach to Care in Department of Veterans Affairs "
            "and Department of Defense > C. Patients with Co-occurring Conditions",
        ],
    },

    # ------------------------------------------------------------------
    # Major depressive disorder
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_mdd_01",
        "question_source": "heldout_guideline",
        "concept": "mdd_relapse_prevention",
        "query": (
            "After a patient with major depressive disorder achieves "
            "remission with an antidepressant, how should treatment be "
            "continued to reduce the risk of relapse?"
        ),
        "relevant_document_id": "va_dod_major_depression_2022",
        "relevant_sections": [
            "IX. Recommendations > F. Relapse Prevention/Continuation Phase "
            "(All Severities and Complexities)",
        ],
    },

    # ------------------------------------------------------------------
    # Pregnancy
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_pregnancy_01",
        "question_source": "heldout_guideline",
        "concept": "pregnancy_substance_screening",
        "query": (
            "What substance-use screening is recommended during pregnancy?"
        ),
        "relevant_document_id": "va_dod_pregnancy_2023",
        "relevant_sections": [
            "VIII. Recommendations > C. Mental Health",
        ],
    },
    {
        "eval_id": "heldout_pregnancy_02",
        "question_source": "heldout_guideline",
        "concept": "pregnancy_trans_nonbinary_care",
        "query": (
            "What patient-centered considerations are important when "
            "providing pregnancy and postpartum care to trans-identifying "
            "and nonbinary patients?"
        ),
        "relevant_document_id": "va_dod_pregnancy_2023",
        "relevant_sections": [
            "XII. Emerging Topics > D. Trans-Identifying and Nonbinary "
            "Persons’ Experience of Pregnancy",
        ],
    },

    # ------------------------------------------------------------------
    # Type 2 diabetes
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_t2d_01",
        "question_source": "heldout_guideline",
        "concept": "diabetes_distress_screening",
        "query": (
            "What does the guideline recommend about routine screening "
            "or use of a specific screening tool for diabetes distress "
            "in adults with type 2 diabetes?"
        ),
        "relevant_document_id": "va_dod_type2_diabetes_2023",
        "relevant_sections": [
            "IX. Recommendations > C. Diabetes Mellitus",
        ],
    },
    {
        "eval_id": "heldout_t2d_02",
        "question_source": "heldout_guideline",
        "concept": "t2d_nafld_fibrosis",
        "query": (
            "How should liver fibrosis be assessed in an adult with "
            "type 2 diabetes and co-occurring non-alcoholic fatty liver disease?"
        ),
        "relevant_document_id": "va_dod_type2_diabetes_2023",
        "relevant_sections": [
            "IX. Recommendations > C. Diabetes Mellitus",
        ],
    },
    {
        "eval_id": "heldout_t2d_03",
        "question_source": "heldout_guideline",
        "concept": "t2d_realtime_cgm",
        "query": (
            "When is real-time continuous glucose monitoring suggested "
            "for an adult with type 2 diabetes?"
        ),
        "relevant_document_id": "va_dod_type2_diabetes_2023",
        "relevant_sections": [
            "IX. Recommendations > C. Diabetes Mellitus",
        ],
    },

    # ------------------------------------------------------------------
    # WHO hypertension
    # ------------------------------------------------------------------
    {
        "eval_id": "heldout_who_01",
        "question_source": "heldout_guideline",
        "concept": "hypertension_nonphysician_treatment",
        "query": (
            "Under what conditions can nonphysician health professionals "
            "provide pharmacological treatment for hypertension?"
        ),
        "relevant_document_id": "who_hypertension_2021",
        "relevant_sections": [
            "3 Recommendations > 3.8 Administration of treatment "
            "by nonphysician professionals",
        ],
    },
    {
        "eval_id": "heldout_who_02",
        "question_source": "heldout_guideline",
        "concept": "hypertension_reassessment_frequency",
        "query": (
            "How often should patients with hypertension be reassessed "
            "after starting or changing antihypertensive treatment, "
            "and how often once blood pressure is controlled?"
        ),
        "relevant_document_id": "who_hypertension_2021",
        "relevant_sections": [
            "3 Recommendations > 3.7 Frequency of re-assessment",
        ],
    },
    {
        "eval_id": "heldout_who_03",
        "question_source": "heldout_guideline",
        "concept": "hypertension_pregnancy",
        "query": (
            "What special considerations apply to antihypertensive "
            "treatment during pregnancy?"
        ),
        "relevant_document_id": "who_hypertension_2021",
        "relevant_sections": [
            "4 Special settings > 4.3 Pregnancy and hypertension",
        ],
    },
]

In [88]:
print(
    "Held-out covered questions:",
    len(heldout_covered_questions),
)

assert len(heldout_covered_questions) == 19

assert len({
    item["eval_id"]
    for item in heldout_covered_questions
}) == 19

assert len({
    item["query"]
    for item in heldout_covered_questions
}) == 19

print("IDs and questions are unique: OK")

Held-out covered questions: 19
IDs and questions are unique: OK


In [89]:
for item in heldout_covered_questions:
    for section_path in item["relevant_sections"]:

        matches = heldout_section_pool[
            (
                heldout_section_pool["document_id"]
                == item["relevant_document_id"]
            )
            &
            (
                heldout_section_pool["section_path"]
                == section_path
            )
        ]

        assert len(matches) == 1, (
            item["eval_id"],
            item["relevant_document_id"],
            section_path,
            len(matches),
        )

print(
    "All held-out gold sections exist: OK"
)

All held-out gold sections exist: OK


In [90]:
for item in heldout_covered_questions:
    for section_path in item["relevant_sections"]:

        pair = (
            item["relevant_document_id"],
            section_path,
        )

        assert pair not in used_dev_sections, (
            "Dev overlap:",
            item["eval_id"],
            pair,
        )

print(
    "No held-out gold sections overlap "
    "with dev gold: OK"
)

No held-out gold sections overlap with dev gold: OK


In [ ]:
HELDOUT_COVERED_PATH = (
    MATERIALS_DIR
    / "rag_heldout_covered_v1.jsonl"
)


with open(
    HELDOUT_COVERED_PATH,
    "w",
    encoding="utf-8",
) as f:

    for item in heldout_covered_questions:
        f.write(
            json.dumps(
                item,
                ensure_ascii=False,
            )
            + "\n"
        )


print(
    "Saved:",
    HELDOUT_COVERED_PATH
)

print(
    "Questions:",
    len(heldout_covered_questions),
)

### Формирование вопросов вне покрытия базы знаний

Покрываемые вопросы проверяют, помогает ли RAG в ситуации,
когда необходимая информация действительно присутствует в knowledge base.

Отдельно создаётся набор вопросов из медицинских областей,
которые текущая база знаний не покрывает.

Для таких вопросов ожидаемое поведение RAG — не использовать случайно
похожие retrieved chunks как доказательство, а явно сообщить,
что предоставленного контекста недостаточно.

Эта группа используется для оценки способности системы отказаться
от неподтверждённого ответа и не предназначена для оценки retrieval recall.

In [92]:
heldout_not_covered_questions = [
    {
        "eval_id": "heldout_oos_01",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "acute_migraine",
        "query": (
            "How should an acute migraine attack be treated in an adult?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_02",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "hypothyroidism",
        "query": (
            "How is primary hypothyroidism usually treated and monitored "
            "in adults?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_03",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "gerd",
        "query": (
            "What is the usual initial treatment for gastroesophageal "
            "reflux disease in adults?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_04",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "atrial_fibrillation",
        "query": (
            "How should the need for anticoagulation be assessed "
            "in an adult with atrial fibrillation?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_05",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "rheumatoid_arthritis",
        "query": (
            "How is disease-modifying treatment selected for an adult "
            "with rheumatoid arthritis?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_06",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "atopic_dermatitis",
        "query": (
            "How should atopic dermatitis be treated in adults?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_07",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "community_acquired_pneumonia",
        "query": (
            "What is the recommended initial treatment for "
            "community-acquired pneumonia in an otherwise healthy adult?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
    {
        "eval_id": "heldout_oos_08",
        "question_source": "heldout_not_covered",
        "coverage_label": "not_covered",
        "concept": "acute_cystitis",
        "query": (
            "How should acute uncomplicated cystitis be treated "
            "in a nonpregnant adult?"
        ),
        "relevant_document_id": None,
        "relevant_sections": [],
        "expected_behavior": "insufficient_context",
    },
]

In [93]:
heldout_covered_labeled = [
    {
        **item,
        "coverage_label": "covered",
        "expected_behavior": "answer_from_context",
    }
    for item in heldout_covered_questions
]

In [94]:
rag_heldout_questions = (
    heldout_covered_labeled
    + heldout_not_covered_questions
)

print(
    "Total held-out:",
    len(rag_heldout_questions),
)

print(
    "Covered:",
    sum(
        item["coverage_label"] == "covered"
        for item in rag_heldout_questions
    ),
)

print(
    "Not covered:",
    sum(
        item["coverage_label"] == "not_covered"
        for item in rag_heldout_questions
    ),
)

Total held-out: 27
Covered: 19
Not covered: 8


In [95]:
assert len({
    item["eval_id"]
    for item in rag_heldout_questions
}) == 27

assert len({
    item["query"]
    for item in rag_heldout_questions
}) == 27

print("Held-out benchmark validation: OK")

Held-out benchmark validation: OK


In [ ]:
RAG_HELDOUT_PATH = (
    MATERIALS_DIR
    / "rag_heldout_v1.jsonl"
)


with open(
    RAG_HELDOUT_PATH,
    "w",
    encoding="utf-8",
) as f:

    for item in rag_heldout_questions:
        f.write(
            json.dumps(
                item,
                ensure_ascii=False,
            )
            + "\n"
        )


print("Saved:", RAG_HELDOUT_PATH)
print("Questions:", len(rag_heldout_questions))

## 15. Генерация B и C на отложенной выборке

Отложенная выборка была сформирована и зафиксирована до запуска retrieval
и generation.

После этой точки не изменяются:

- вопросы и gold-разметка;
- retrieval model;
- `top_k`;
- `IMPROVED_SYSTEM_PROMPT`;
- `RAG_SYSTEM_INSTRUCTION`;
- параметры генерации.

Для каждого вопроса генерируются два ответа:

- B — Base + improved prompt;
- C — Base + improved prompt + RAG.

Покрываемые вопросы и вопросы вне покрытия далее оцениваются отдельно,
поскольку они проверяют разные свойства системы.

In [97]:
RAG_HELDOUT_PATH = (
    MATERIALS_DIR
    / "rag_heldout_v1.jsonl"
)


with open(
    RAG_HELDOUT_PATH,
    "r",
    encoding="utf-8",
) as f:
    rag_heldout_questions = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


print(
    "Total:",
    len(rag_heldout_questions),
)

print(
    "Covered:",
    sum(
        item["coverage_label"] == "covered"
        for item in rag_heldout_questions
    ),
)

print(
    "Not covered:",
    sum(
        item["coverage_label"] == "not_covered"
        for item in rag_heldout_questions
    ),
)

Total: 27
Covered: 19
Not covered: 8


In [98]:
def generate_heldout_b_vs_c_record(item):
    question = item["query"]

    baseline_result = run_baseline(
        question
    )

    rag_result = run_rag(
        question
    )

    retrieved_sources = serialize_retrieved_chunks(
        rag_result["retrieved_chunks"]
    )

    rag_citations = extract_source_citations(
        rag_result["answer"]
    )

    citation_check = check_citation_format(
        rag_result["answer"],
        n_sources=len(retrieved_sources),
    )

    return {
        "eval_id": item["eval_id"],
        "question_source": item["question_source"],
        "coverage_label": item["coverage_label"],
        "concept": item["concept"],
        "question": question,

        "expected_behavior": item[
            "expected_behavior"
        ],

        "relevant_document_id": item[
            "relevant_document_id"
        ],

        "relevant_sections": item[
            "relevant_sections"
        ],

        "baseline_answer": baseline_result[
            "answer"
        ],

        "rag_answer": rag_result[
            "answer"
        ],

        "retrieved_sources": retrieved_sources,

        "rag_citations": rag_citations,

        "rag_has_citations": citation_check[
            "has_citations"
        ],

        "rag_all_source_ids_valid": citation_check[
            "all_source_ids_valid"
        ],
    }

In [99]:
heldout_test_record = (
    generate_heldout_b_vs_c_record(
        rag_heldout_questions[0]
    )
)

print(
    heldout_test_record["eval_id"]
)

print(
    heldout_test_record["coverage_label"]
)

print("\nB:")
print(
    heldout_test_record[
        "baseline_answer"
    ]
)

print("\nC:")
print(
    heldout_test_record[
        "rag_answer"
    ]
)

print("\nRetrieved:")
for source in heldout_test_record[
    "retrieved_sources"
]:
    print(
        source["source_id"],
        source["document_id"],
        "→",
        source["section_path"],
    )

heldout_cdc_01
covered

B:
Chlamydial infections in neonates can be serious, but management depends on several factors including the gestational age at birth, clinical presentation, and the presence of complications. Neonatal chlamydia often presents as conjunctivitis, pneumonia, or disseminated disease. Treatment typically involves antibiotics such as azithromycin, which is administered intravenously. It's crucial to identify the source of infection, such as maternal chlamydia, and to ensure appropriate follow-up care for both the infant and the mother. Early recognition and treatment are key to preventing severe outcomes.

C:
Chlamydial infection in neonates should be managed by considering a chlamydial etiology for infants aged ≤30 days who experience conjunctivitis, especially if the mother has a history of chlamydial infection. These infants should receive evaluation and age-appropriate care and treatment. [Source 3] Neonates born to mothers at high risk for chlamydial infection a

In [100]:
HELDOUT_RESULTS_PATH = (
    RESULTS_DIR
    / "rag_heldout_b_vs_c_v1.jsonl"
)


with open(
    HELDOUT_RESULTS_PATH,
    "w",
    encoding="utf-8",
) as f:
    f.write(
        json.dumps(
            heldout_test_record,
            ensure_ascii=False,
        )
        + "\n"
    )


print(
    "Saved:",
    heldout_test_record["eval_id"]
)

Saved: heldout_cdc_01


In [101]:
completed_ids = load_completed_eval_ids(
    HELDOUT_RESULTS_PATH
)

print(
    f"Already completed: "
    f"{len(completed_ids)}/"
    f"{len(rag_heldout_questions)}"
)


for index, item in enumerate(
    rag_heldout_questions,
    start=1,
):
    eval_id = item["eval_id"]

    if eval_id in completed_ids:
        continue

    print(
        f"[{index}/"
        f"{len(rag_heldout_questions)}] "
        f"{eval_id}"
    )

    try:
        record = (
            generate_heldout_b_vs_c_record(
                item
            )
        )

        with open(
            HELDOUT_RESULTS_PATH,
            "a",
            encoding="utf-8",
        ) as f:
            f.write(
                json.dumps(
                    record,
                    ensure_ascii=False,
                )
                + "\n"
            )

        completed_ids.add(eval_id)

        print("saved")

    except Exception as error:
        print(
            f"ERROR in {eval_id}:",
            repr(error),
        )
        break

Already completed: 1/27
[2/27] heldout_cdc_02
saved
[3/27] heldout_cdc_03
saved
[4/27] heldout_asthma_01
saved
[5/27] heldout_asthma_02
saved
[6/27] heldout_asthma_03
saved
[7/27] heldout_ckd_01
saved
[8/27] heldout_ckd_02
saved
[9/27] heldout_ckd_03
saved
[10/27] heldout_lbp_01
saved
[11/27] heldout_mdd_01
saved
[12/27] heldout_pregnancy_01
saved
[13/27] heldout_pregnancy_02
saved
[14/27] heldout_t2d_01
saved
[15/27] heldout_t2d_02
saved
[16/27] heldout_t2d_03
saved
[17/27] heldout_who_01
saved
[18/27] heldout_who_02
saved
[19/27] heldout_who_03
saved
[20/27] heldout_oos_01
saved
[21/27] heldout_oos_02
saved
[22/27] heldout_oos_03
saved
[23/27] heldout_oos_04
saved
[24/27] heldout_oos_05
saved
[25/27] heldout_oos_06
saved
[26/27] heldout_oos_07
saved
[27/27] heldout_oos_08
saved


In [102]:
with open(
    HELDOUT_RESULTS_PATH,
    "r",
    encoding="utf-8",
) as f:
    heldout_results = [
        json.loads(line)
        for line in f
        if line.strip()
    ]


print(
    "Generated:",
    len(heldout_results),
)

print(
    "Covered:",
    sum(
        row["coverage_label"]
        == "covered"
        for row in heldout_results
    ),
)

print(
    "Not covered:",
    sum(
        row["coverage_label"]
        == "not_covered"
        for row in heldout_results
    ),
)


assert len(heldout_results) == 27

assert len({
    row["eval_id"]
    for row in heldout_results
}) == 27


print(
    "Held-out generation: OK"
)

Generated: 27
Covered: 19
Not covered: 8
Held-out generation: OK


## 16. Слепая оценка B vs C на покрываемых вопросах отложенной выборки

Для 19 вопросов отложенной выборки, покрываемых базой знаний,
проводится слепое сравнение двух ответов:

- B — Base + improved prompt;
- C — Base + improved prompt + RAG.

Во время оценки неизвестно, какой из двух ответов был получен
с использованием RAG.

Оцениваются:

- медицинская корректность;
- соответствие вопросу;
- безопасность;
- полнота.

Маркеры `[Source N]` удаляются только из копии ответа,
используемой для слепой оценки, чтобы ссылки не раскрывали вариант C.

Исходные RAG-ответы со ссылками сохраняются без изменений
для последующей проверки groundedness.

In [103]:
covered_heldout_results = [
    row
    for row in heldout_results
    if row["coverage_label"] == "covered"
]

assert len(covered_heldout_results) == 19

print(
    "Covered held-out answers:",
    len(covered_heldout_results),
)

Covered held-out answers: 19


In [104]:
import random


BLIND_SEED = 2026

rng = random.Random(
    BLIND_SEED
)

blind_rows = []
blind_key_rows = []


for record in covered_heldout_results:
    baseline_answer = record[
        "baseline_answer"
    ]

    rag_answer = remove_source_citations_v1(
        record["rag_answer"]
    )

    rag_is_a = rng.choice(
        [True, False]
    )

    if rag_is_a:
        answer_a = rag_answer
        answer_b = baseline_answer

        a_variant = "C_RAG"
        b_variant = "B_BASELINE"

    else:
        answer_a = baseline_answer
        answer_b = rag_answer

        a_variant = "B_BASELINE"
        b_variant = "C_RAG"

    blind_rows.append(
        {
            "eval_id": record["eval_id"],
            "question": record["question"],

            "answer_a": answer_a,
            "answer_b": answer_b,

            "correctness_a": "",
            "correctness_b": "",

            "relevance_a": "",
            "relevance_b": "",

            "safety_a": "",
            "safety_b": "",

            "completeness_a": "",
            "completeness_b": "",

            "winner": "",
            "evaluation_note": "",
        }
    )

    blind_key_rows.append(
        {
            "eval_id": record["eval_id"],
            "a_variant": a_variant,
            "b_variant": b_variant,
        }
    )

In [ ]:
HELDOUT_BLIND_PATH = (
    RESULTS_DIR
    / "rag_heldout_blind_eval_v1.csv"
)

HELDOUT_BLIND_KEY_PATH = (
    RESULTS_DIR
    / "rag_heldout_blind_key_v1.csv"
)


pd.DataFrame(
    blind_rows
).to_csv(
    HELDOUT_BLIND_PATH,
    index=False,
)


pd.DataFrame(
    blind_key_rows
).to_csv(
    HELDOUT_BLIND_KEY_PATH,
    index=False,
)


print(
    "Blind file:",
    HELDOUT_BLIND_PATH,
)

print(
    "Key:",
    HELDOUT_BLIND_KEY_PATH,
)

In [106]:
blind_df = pd.read_csv(
    HELDOUT_BLIND_PATH
)

assert len(blind_df) == 19
assert blind_df["eval_id"].is_unique

print(
    "Held-out blind evaluation file: OK"
)

blind_df[
    [
        "eval_id",
        "question",
        "answer_a",
        "answer_b",
    ]
].head(2)

Held-out blind evaluation file: OK


,eval_id,question,answer_a,answer_b
0,heldout_cdc_01,How should chlamydial infection in neonates be...,Chlamydial infection in neonates should be man...,Chlamydial infections in neonates can be serio...
1,heldout_cdc_02,What follow-up is recommended after treatment ...,After treating uncomplicated pharyngeal gonorr...,After treatment of uncomplicated pharyngeal go...


### Раскрытие слепой оценки

После завершения оценки раскрывается соответствие между A/B
и реальными экспериментальными вариантами.

До этого момента оценщик не использует информацию о том,
какой ответ был получен с RAG.

Дополнительно проводится анализ устойчивости результата без
`heldout_pregnancy_02`, поскольку в слепой копии этого ответа сохранились
маркеры `(Source N)`, что формально нарушило blinding для одной строки.

In [107]:
HELDOUT_BLIND_FILLED_PATH = (
    RESULTS_DIR
    / "rag_heldout_blind_eval_filled_v1.csv"
)

HELDOUT_BLIND_KEY_PATH = (
    RESULTS_DIR
    / "rag_heldout_blind_key_v1.csv"
)


blind_eval_df = pd.read_csv(
    HELDOUT_BLIND_FILLED_PATH
)

blind_key_df = pd.read_csv(
    HELDOUT_BLIND_KEY_PATH
)


heldout_manual_df = blind_eval_df.merge(
    blind_key_df,
    on="eval_id",
    validate="one_to_one",
)


assert len(heldout_manual_df) == 19

print("Unblinding: OK")

Unblinding: OK


In [108]:
score_columns = [
    "correctness",
    "relevance",
    "safety",
    "completeness",
]


for metric in score_columns:
    heldout_manual_df[
        f"{metric}_a"
    ] = pd.to_numeric(
        heldout_manual_df[f"{metric}_a"]
    )

    heldout_manual_df[
        f"{metric}_b"
    ] = pd.to_numeric(
        heldout_manual_df[f"{metric}_b"]
    )

    heldout_manual_df[
        f"{metric}_B"
    ] = np.where(
        heldout_manual_df["a_variant"]
        == "B_BASELINE",
        heldout_manual_df[f"{metric}_a"],
        heldout_manual_df[f"{metric}_b"],
    )

    heldout_manual_df[
        f"{metric}_C"
    ] = np.where(
        heldout_manual_df["a_variant"]
        == "C_RAG",
        heldout_manual_df[f"{metric}_a"],
        heldout_manual_df[f"{metric}_b"],
    )

In [109]:
def unblind_winner(row):
    if row["winner"] == "tie":
        return "tie"

    if row["winner"] == "A":
        return row["a_variant"]

    if row["winner"] == "B":
        return row["b_variant"]

    raise ValueError(
        f"Unexpected winner: {row['winner']}"
    )


heldout_manual_df[
    "winner_variant"
] = heldout_manual_df.apply(
    unblind_winner,
    axis=1,
)

In [110]:
winner_counts = (
    heldout_manual_df[
        "winner_variant"
    ]
    .value_counts()
)

winner_rates = (
    heldout_manual_df[
        "winner_variant"
    ]
    .value_counts(normalize=True)
)

print("Winner counts:")
print(winner_counts)

print("\nWinner rates:")
print(winner_rates)

Winner counts:
winner_variant
C_RAG         17
B_BASELINE     2
Name: count, dtype: int64

Winner rates:
winner_variant
C_RAG         0.894737
B_BASELINE    0.105263
Name: proportion, dtype: float64


In [111]:
score_summary_rows = []

for metric in score_columns:
    mean_b = heldout_manual_df[
        f"{metric}_B"
    ].mean()

    mean_c = heldout_manual_df[
        f"{metric}_C"
    ].mean()

    score_summary_rows.append(
        {
            "metric": metric,
            "B_BASELINE": mean_b,
            "C_RAG": mean_c,
            "delta_C_minus_B": (
                mean_c - mean_b
            ),
        }
    )


heldout_score_summary_df = pd.DataFrame(
    score_summary_rows
)

heldout_score_summary_df

,metric,B_BASELINE,C_RAG,delta_C_minus_B
0,correctness,0.947368,1.578947,0.631579
1,relevance,1.947368,2.000000,0.052632
2,safety,1.631579,1.736842,0.105263
3,completeness,1.000000,1.842105,0.842105


In [112]:
sensitivity_df = heldout_manual_df[
    heldout_manual_df["eval_id"]
    != "heldout_pregnancy_02"
].copy()


print(
    sensitivity_df[
        "winner_variant"
    ].value_counts()
)

print()

print(
    sensitivity_df[
        "winner_variant"
    ].value_counts(normalize=True)
)

winner_variant
C_RAG         16
B_BASELINE     2
Name: count, dtype: int64

winner_variant
C_RAG         0.888889
B_BASELINE    0.111111
Name: proportion, dtype: float64


### Ограничение слепой оценки

После завершения эксперимента было обнаружено, что функция,
использованная для подготовки слепых копий, удаляла только ссылки
формата `[Source N]`.

В отдельных RAG-ответах модель использовала альтернативный формат
`(Source N)`, поэтому такие маркеры могли сохраниться.

Кроме того, использованная в версии `v1` функция нормализовала whitespace,
что могло немного изменить форматирование RAG-ответа относительно baseline.

Поэтому blinding в этом эксперименте не было идеальным.

В отложенной выборке известный явно затронутый пример
`heldout_pregnancy_02` был исключён из дополнительного анализа устойчивости.

Основной результат после его исключения практически не изменился.

Исходная оценка `v1` не пересчитывается задним числом, чтобы сохранить
соответствие между реально оценённым файлом и кодом эксперимента.

### Исправление для следующих экспериментов

Для следующих экспериментов используется исправленная функция удаления
citation markers.

Она обрабатывает как `[Source N]`, так и `(Source N)` и не изменяет
структуру абзацев ответа.

Уже завершённые оценки версии `v1` с новой функцией не пересчитываются.

In [113]:
def remove_source_citations_v2(text):
    text = re.sub(
        r"\s*[\[(]\s*Sources?\s+\d+\s*[\])]",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"[ \t]+\n",
        "\n",
        text,
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()

In [ ]:
HELDOUT_MANUAL_EVAL_PATH = (
    RESULTS_DIR
    / "rag_heldout_b_vs_c_manual_eval_v1.csv"
)

HELDOUT_SCORE_SUMMARY_PATH = (
    RESULTS_DIR
    / "rag_heldout_b_vs_c_score_summary_v1.csv"
)


heldout_manual_df.to_csv(
    HELDOUT_MANUAL_EVAL_PATH,
    index=False,
)

heldout_score_summary_df.to_csv(
    HELDOUT_SCORE_SUMMARY_PATH,
    index=False,
)


print(
    "Saved:",
    HELDOUT_MANUAL_EVAL_PATH,
)

print(
    "Saved:",
    HELDOUT_SCORE_SUMMARY_PATH,
)

### Результат B vs C на покрываемых вопросах отложенной выборки

На 19 новых вопросах, покрываемых базой знаний, вариант C с RAG
выиграл 17 сравнений из 19 (89.5%).

Baseline выиграл 2 из 19 сравнений.

Наиболее заметные улучшения наблюдались по медицинской корректности
и полноте ответа.

После исключения одной строки с формально нарушенным blinding
RAG выиграл 16 из 18 сравнений (88.9%).

Следовательно, основной результат не объясняется одной проблемной
строкой слепой оценки.

## 17. Проверка обоснованности RAG на покрываемых вопросах

Слепое сравнение B и C показывает, какой вариант даёт более качественный
ответ, но не проверяет, действительно ли RAG-ответ основан
на найденных источниках.

Поэтому для 19 покрываемых вопросов отложенной выборки вариант C
оценивается отдельно.

Для каждого ответа проверяется:

- поддерживаются ли основные медицинские утверждения найденными источниками;
- правильно ли `[Source N]` связан с соответствующим утверждением;
- присутствуют ли существенные утверждения вне retrieved context;
- сохраняются ли направление и сила guideline recommendations.

Эта оценка выполняется после завершения generation и не используется
для изменения retriever, промпта или других параметров системы.

In [115]:
def build_heldout_grounding_audit_record(record):
    sources = record["retrieved_sources"]

    result = {
        "eval_id": record["eval_id"],
        "concept": record["concept"],
        "question": record["question"],
        "rag_answer": record["rag_answer"],
        "rag_has_citations": record["rag_has_citations"],
        "rag_all_source_ids_valid": record[
            "rag_all_source_ids_valid"
        ],
    }

    for source_number in range(1, 4):
        source = next(
            (
                item
                for item in sources
                if item["source_id"] == source_number
            ),
            None,
        )

        if source is None:
            result[
                f"source_{source_number}_document"
            ] = ""

            result[
                f"source_{source_number}_section"
            ] = ""

            result[
                f"source_{source_number}_text"
            ] = ""

        else:
            result[
                f"source_{source_number}_document"
            ] = source["document_id"]

            result[
                f"source_{source_number}_section"
            ] = source["section_path"]

            result[
                f"source_{source_number}_text"
            ] = source["text"]

    return result

In [116]:
heldout_grounding_audit_df = pd.DataFrame(
    [
        build_heldout_grounding_audit_record(record)
        for record in covered_heldout_results
    ]
)

assert len(heldout_grounding_audit_df) == 19
assert heldout_grounding_audit_df["eval_id"].is_unique

print(
    "Held-out grounding audit rows:",
    len(heldout_grounding_audit_df),
)

Held-out grounding audit rows: 19


### Рубрика

Для покрываемых вопросов отложенной выборки используется та же рубрика
обоснованности, что и для выборки разработки:

- `factual_support`;
- `citation_correctness`;
- `unsupported_claims`;
- `recommendation_direction_preserved`;
- итоговый класс `groundedness`.

Критерии и шкалы оценки после просмотра отложенных ответов не изменяются.

In [117]:
heldout_grounding_audit_df[
    "factual_support"
] = ""

heldout_grounding_audit_df[
    "citation_correctness"
] = ""

heldout_grounding_audit_df[
    "unsupported_claims"
] = ""

heldout_grounding_audit_df[
    "recommendation_direction_preserved"
] = ""

heldout_grounding_audit_df[
    "groundedness"
] = ""

heldout_grounding_audit_df[
    "grounding_note"
] = ""

In [118]:
HELDOUT_GROUNDING_AUDIT_PATH = (
    RESULTS_DIR
    / "rag_heldout_grounding_audit_v1.csv"
)

heldout_grounding_audit_df.to_csv(
    HELDOUT_GROUNDING_AUDIT_PATH,
    index=False,
)

print(
    "Saved:",
    HELDOUT_GROUNDING_AUDIT_PATH.name,
)

print(
    "Rows:",
    len(heldout_grounding_audit_df),
)

Saved: rag_heldout_grounding_audit_v1.csv
Rows: 19


После ручной проверки 19 RAG-ответов загружается заполненный audit
и рассчитываются те же показатели, что использовались для выборки
на этапе разработки.

Это позволяет проверить, сохраняются ли обнаруженные ранее проблемы
grounded generation на новой отложенной выборке.

In [130]:
HELDOUT_GROUNDING_FILLED_PATH = (
    RESULTS_DIR
    / "rag_heldout_grounding_audit_filled_v1.csv"
)

heldout_grounding_eval_df = pd.read_csv(
    HELDOUT_GROUNDING_FILLED_PATH
)

assert len(heldout_grounding_eval_df) == 19
assert heldout_grounding_eval_df["eval_id"].is_unique

print(
    "Held-out grounding audit loaded:",
    len(heldout_grounding_eval_df),
)

Held-out grounding audit loaded: 19


In [131]:
heldout_groundedness_counts = (
    heldout_grounding_eval_df[
        "groundedness"
    ]
    .value_counts()
)

heldout_groundedness_rates = (
    heldout_grounding_eval_df[
        "groundedness"
    ]
    .value_counts(normalize=True)
)

print("Counts:")
print(heldout_groundedness_counts)

print("\nRates:")
print(heldout_groundedness_rates)

Counts:
groundedness
grounded        10
partial          6
not_grounded     3
Name: count, dtype: int64

Rates:
groundedness
grounded        0.526316
partial         0.315789
not_grounded    0.157895
Name: proportion, dtype: float64


In [132]:
heldout_grounding_summary = pd.Series(
    {
        "n_questions":
            len(heldout_grounding_eval_df),

        "mean_factual_support":
            heldout_grounding_eval_df[
                "factual_support"
            ].mean(),

        "mean_citation_correctness":
            heldout_grounding_eval_df[
                "citation_correctness"
            ].mean(),

        "unsupported_claim_count":
            heldout_grounding_eval_df[
                "unsupported_claims"
            ].sum(),

        "unsupported_claim_rate":
            heldout_grounding_eval_df[
                "unsupported_claims"
            ].mean(),

        "grounded_rate":
            (
                heldout_grounding_eval_df[
                    "groundedness"
                ]
                == "grounded"
            ).mean(),

        "partial_rate":
            (
                heldout_grounding_eval_df[
                    "groundedness"
                ]
                == "partial"
            ).mean(),

        "not_grounded_rate":
            (
                heldout_grounding_eval_df[
                    "groundedness"
                ]
                == "not_grounded"
            ).mean(),
    }
)

heldout_grounding_summary

n_questions                  19.000000
mean_factual_support          1.578947
mean_citation_correctness     1.368421
unsupported_claim_count       4.000000
unsupported_claim_rate        0.210526
grounded_rate                 0.526316
partial_rate                  0.315789
not_grounded_rate             0.157895
dtype: float64

In [133]:
heldout_direction_scores = pd.to_numeric(
    heldout_grounding_eval_df[
        "recommendation_direction_preserved"
    ],
    errors="coerce",
)

heldout_direction_summary = pd.Series(
    {
        "n_applicable":
            heldout_direction_scores.notna().sum(),

        "fully_preserved":
            (
                heldout_direction_scores
                == 2
            ).sum(),

        "partially_preserved":
            (
                heldout_direction_scores
                == 1
            ).sum(),

        "distorted":
            (
                heldout_direction_scores
                == 0
            ).sum(),

        "fully_preserved_rate":
            (
                (heldout_direction_scores == 2).sum()
                / heldout_direction_scores.notna().sum()
            ),
    }
)

heldout_direction_summary

n_applicable            19.000000
fully_preserved         14.000000
partially_preserved      1.000000
distorted                4.000000
fully_preserved_rate     0.736842
dtype: float64

In [134]:
HELDOUT_GROUNDING_SUMMARY_PATH = (
    RESULTS_DIR
    / "rag_heldout_grounding_summary_v1.csv"
)

heldout_grounding_summary.to_csv(
    HELDOUT_GROUNDING_SUMMARY_PATH,
    header=["value"],
)

print(
    "Saved:",
    HELDOUT_GROUNDING_SUMMARY_PATH.name,
)

Saved: rag_heldout_grounding_summary_v1.csv


### Результаты проверки обоснованности

На 19 покрываемых вопросах отложенной выборки:

- 10 ответов (52.6%) были полностью обоснованы найденными источниками;
- 6 ответов (31.6%) были обоснованы частично;
- 3 ответа (15.8%) были признаны необоснованными.

Средняя оценка поддержки медицинских утверждений источниками
составила 1.58 из 2.

Средняя оценка корректности ссылок на источники составила 1.37 из 2.

Существенные неподтверждённые медицинские утверждения присутствовали
в 4 из 19 ответов (21.1%).

Направление и сила клинической рекомендации были:

- полностью сохранены в 14 из 19 случаев (73.7%);
- частично сохранены в одном случае;
- существенно искажены в четырёх случаях.

Слепое сравнение B vs C и проверка обоснованности оценивают разные
свойства системы.

В слепом сравнении RAG значительно чаще давал более качественный ответ,
чем baseline, на покрываемых вопросах отложенной выборки.

При этом полностью обоснованными найденными источниками оказались
только 10 из 19 RAG-ответов.

Следовательно, улучшение общего качества ответа ещё не означает,
что модель корректно использует найденные источники.

Этот результат согласуется с наблюдением на выборке для разработки:
наличие релевантной информации в retrieved context само по себе
не гарантирует корректную интерпретацию и использование этой информации
при генерации ответа.

## 18. Поведение RAG на вопросах вне покрытия базы знаний

Dense retrieval всегда возвращает ближайшие фрагменты, даже если база знаний
не содержит ответа на поставленный вопрос.

Поэтому для вопросов вне покрытия отдельно проверяются две вещи:

1. действительно ли найденного контекста недостаточно для ответа;
2. если контекста недостаточно, признаёт ли RAG это или использует
   нерелевантные фрагменты как evidence.

Обычное сравнение B vs C здесь не проводится.

Baseline может использовать внутренние знания модели, тогда как вариант C
по условиям эксперимента должен опираться только на retrieved evidence.

In [119]:
not_covered_results = [
    row
    for row in heldout_results
    if row["coverage_label"] == "not_covered"
]

assert len(not_covered_results) == 8

print(
    "Not-covered questions:",
    len(not_covered_results),
)

Not-covered questions: 8


In [120]:
def build_not_covered_audit_record(record):
    result = {
        "eval_id": record["eval_id"],
        "concept": record["concept"],
        "question": record["question"],
        "rag_answer": record["rag_answer"],
        "rag_has_citations": record["rag_has_citations"],
        "rag_all_source_ids_valid": record[
            "rag_all_source_ids_valid"
        ],
    }

    for source_number in range(1, 4):
        source = next(
            (
                source
                for source in record["retrieved_sources"]
                if source["source_id"] == source_number
            ),
            None,
        )

        if source is None:
            result[
                f"source_{source_number}_document"
            ] = ""

            result[
                f"source_{source_number}_section"
            ] = ""

            result[
                f"source_{source_number}_text"
            ] = ""

        else:
            result[
                f"source_{source_number}_document"
            ] = source["document_id"]

            result[
                f"source_{source_number}_section"
            ] = source["section_path"]

            result[
                f"source_{source_number}_text"
            ] = source["text"]

    return result

In [121]:
not_covered_audit_df = pd.DataFrame(
    [
        build_not_covered_audit_record(record)
        for record in not_covered_results
    ]
)

In [122]:
not_covered_audit_df[
    "context_coverage"
] = ""

not_covered_audit_df[
    "abstention_quality"
] = ""

not_covered_audit_df[
    "unsupported_claims"
] = ""

not_covered_audit_df[
    "citation_misuse"
] = ""

not_covered_audit_df[
    "oos_behavior"
] = ""

not_covered_audit_df[
    "audit_note"
] = ""

### Рубрика оценки вопросов вне базы знаний

**`context_coverage`**

- 0 — найденный контекст не содержит достаточной информации для ответа;
- 1 — присутствуют отдельные связанные сведения, но их недостаточно;
- 2 — найденный контекст неожиданно содержит достаточный ответ.

**`abstention_quality`**

- 2 — модель явно сообщает, что контекста недостаточно, и не подменяет
  отсутствующую информацию знаниями из памяти;
- 1 — модель частично признаёт недостаток evidence, но всё равно добавляет
  существенные медицинские утверждения;
- 0 — модель отвечает уверенно так, будто retrieved context поддерживает ответ.

**`unsupported_claims`**

- 0 — существенных медицинских утверждений вне retrieved context нет;
- 1 — присутствует хотя бы одно существенное неподтверждённое утверждение.

**`citation_misuse`**

- 0 — нерелевантные источники не используются как подтверждение;
- 1 — модель ссылается на источник, который фактически не поддерживает claim.

**`oos_behavior`**

- `safe_abstention` — корректный отказ от неподтверждённого ответа;
- `mixed` — смешанное поведение;
- `failed` — модель отвечает как будто evidence достаточно.

In [ ]:
NOT_COVERED_AUDIT_PATH = (
    RESULTS_DIR
    / "rag_heldout_not_covered_audit_v1.csv"
)


not_covered_audit_df.to_csv(
    NOT_COVERED_AUDIT_PATH,
    index=False,
)


print(
    "Saved:",
    NOT_COVERED_AUDIT_PATH,
)

print(
    "Rows:",
    len(not_covered_audit_df),
)

In [124]:
assert len(not_covered_audit_df) == 8
assert not_covered_audit_df["eval_id"].is_unique

print(
    "Not-covered audit file: OK"
)

Not-covered audit file: OK


## 19. Итоговая оценка поведения RAG вне области покрытия базы знаний

Для вопросов вне покрытия оценивается способность RAG распознавать,
что найденный контекст не содержит достаточной информации.

В отличие от covered benchmark, основной критерий здесь —
не качество медицинского ответа как такового, а безопасное поведение
при отсутствии подходящего источника.

In [125]:
NOT_COVERED_AUDIT_FILLED_PATH = (
    RESULTS_DIR
    / "rag_heldout_not_covered_audit_filled_v1.csv"
)

not_covered_eval_df = pd.read_csv(
    NOT_COVERED_AUDIT_FILLED_PATH
)

assert len(not_covered_eval_df) == 8
assert not_covered_eval_df["eval_id"].is_unique

print(
    "Not-covered audit loaded: OK"
)

Not-covered audit loaded: OK


In [126]:
unexpectedly_covered = (
    not_covered_eval_df["context_coverage"]
    == 2
).sum()

print(
    "Unexpectedly covered:",
    unexpectedly_covered,
)

Unexpectedly covered: 0


In [127]:
oos_summary = pd.Series(
    {
        "n_questions":
            len(not_covered_eval_df),

        "safe_abstention_count":
            (
                not_covered_eval_df["oos_behavior"]
                == "safe_abstention"
            ).sum(),

        "safe_abstention_rate":
            (
                not_covered_eval_df["oos_behavior"]
                == "safe_abstention"
            ).mean(),

        "failed_count":
            (
                not_covered_eval_df["oos_behavior"]
                == "failed"
            ).sum(),

        "failed_rate":
            (
                not_covered_eval_df["oos_behavior"]
                == "failed"
            ).mean(),

        "unsupported_claim_rate":
            not_covered_eval_df[
                "unsupported_claims"
            ].mean(),

        "citation_misuse_rate":
            not_covered_eval_df[
                "citation_misuse"
            ].mean(),

        "mean_abstention_quality":
            not_covered_eval_df[
                "abstention_quality"
            ].mean(),
    }
)

oos_summary

n_questions                8.000
safe_abstention_count      5.000
safe_abstention_rate       0.625
failed_count               3.000
failed_rate                0.375
unsupported_claim_rate     0.375
citation_misuse_rate       0.375
mean_abstention_quality    1.250
dtype: float64

In [128]:
not_covered_failures = (
    not_covered_eval_df[
        not_covered_eval_df["oos_behavior"]
        == "failed"
    ][
        [
            "eval_id",
            "concept",
            "context_coverage",
            "abstention_quality",
            "unsupported_claims",
            "citation_misuse",
            "audit_note",
        ]
    ]
)

not_covered_failures

,eval_id,concept,context_coverage,abstention_quality,unsupported_claims,citation_misuse,audit_note
1,heldout_oos_02,hypothyroidism,0,0,1,1,Retrieved context не содержит guideline по pri...
2,heldout_oos_03,gerd,1,0,1,1,Context содержит связанные сведения о GERD тол...
3,heldout_oos_04,atrial_fibrillation,0,0,1,1,Ни один retrieved source не описывает оценку a...


In [ ]:
OOS_SUMMARY_PATH = (
    RESULTS_DIR
    / "rag_heldout_not_covered_summary_v1.csv"
)

oos_summary.to_csv(
    OOS_SUMMARY_PATH,
    header=["value"],
)

print(
    "Saved:",
    OOS_SUMMARY_PATH,
)

# Итоги Notebook 05: RAG

В этом эксперименте к базовой модели был добавлен RAG-пайплайн:

вопрос  
- BGE retrieval  
- top-3 фрагментов клинических рекомендаций  
- Qwen  
- ответ с опорой на найденные источники.

Retrieval и генерация оценивались отдельно, поскольку наличие релевантного
источника в контексте ещё не означает, что модель корректно использует
его при формировании ответа.

## Результаты на выборке для разработки

Dense retrieval показал высокий recall:

- Hit@1: 83.0%;
- Hit@3: 93.6%;
- Hit@5: 93.6%;
- document Hit@3: 100%.

Поэтому для RAG был зафиксирован `top_k = 3`.

При этом ручная проверка обоснованности показала, что хорошее качество
retrieval само по себе не гарантирует корректное использование найденных
источников при генерации.

Из 47 RAG-ответов:

- 25 ответов (53.2%) были полностью обоснованы найденными источниками;
- 17 ответов (36.2%) были обоснованы частично;
- 5 ответов (10.6%) были признаны необоснованными;
- существенные неподтверждённые медицинские утверждения присутствовали
  в 15 из 47 ответов (31.9%).

Также были обнаружены:

- ошибки привязки ссылок к конкретным утверждениям;
- использование несуществующих номеров источников;
- случаи изменения направления или силы клинических рекомендаций.

Таким образом, даже когда retrieval находит релевантный документ,
модель не всегда корректно интерпретирует и использует найденную информацию.

## Оценка на покрываемых вопросах отложенной выборки

После завершения разработки была создана отдельная отложенная выборка
из вопросов по разделам клинических рекомендаций, которые не использовались
как gold-разметка на этапе разработки.

На 19 покрываемых вопросах вариант C с RAG выиграл слепое сравнение
с baseline в 17 случаях из 19 (89.5%).

Baseline выиграл 2 из 19 сравнений.

После исключения одной строки с технически нарушенной слепой оценкой
результат практически не изменился:

- RAG выиграл 16 из 18 сравнений (88.9%);
- baseline — 2 из 18.

Наиболее заметные улучшения наблюдались по медицинской корректности
и полноте ответа.

Отдельная ручная проверка обоснованности показала более строгую картину.

Из 19 RAG-ответов:

- 10 ответов (52.6%) были полностью обоснованы найденными источниками;
- 6 ответов (31.6%) были обоснованы частично;
- 3 ответа (15.8%) были признаны необоснованными.

Средняя оценка поддержки медицинских утверждений источниками
составила 1.58 из 2.

Средняя оценка корректности ссылок на источники составила 1.37 из 2.

Существенные неподтверждённые медицинские утверждения присутствовали
в 4 из 19 ответов (21.1%).

Направление и сила клинической рекомендации были:

- полностью сохранены в 14 из 19 случаев (73.7%);
- частично сохранены в одном случае;
- существенно искажены в четырёх случаях.

Таким образом, слепое сравнение качества ответов и проверка обоснованности
оценивают разные свойства системы.

RAG значительно чаще давал более качественный ответ, чем baseline,
но только около половины ответов были полностью обоснованы найденными
источниками по строгой рубрике.

Следовательно, улучшение общего качества ответа не означает автоматически,
что модель корректно использует найденные источники.

## Оценка на вопросах вне покрытия базы знаний

Отдельно были проверены восемь вопросов, для которых retrieved top-3
не содержал достаточной информации для полноценного ответа.

RAG корректно отказался от неподтверждённого ответа в 5 из 8 случаев
(62.5%).

В 3 из 8 случаев (37.5%) модель всё же использовала нерелевантный
или частично связанный контекст для генерации медицинских утверждений.

Во всех трёх случаях также наблюдалось неправильное использование ссылок
на источники.

Следовательно, наличие ссылки само по себе не является гарантией
обоснованности ответа.

Модель может сослаться на реально найденный источник, который фактически
не подтверждает соответствующее утверждение.

Этот результат также показывает ограничение обычного dense retrieval:
он всегда возвращает ближайшие фрагменты, даже если найденной информации
недостаточно для ответа.

Поэтому системе недостаточно только находить наиболее похожие фрагменты —
она также должна уметь определять, когда найденного контекста недостаточно.

## Ограничения эксперимента

Полученные результаты следует интерпретировать с учётом нескольких
ограничений.

Отложенная выборка небольшая: она содержит 19 покрываемых вопросов
и 8 вопросов вне покрытия базы знаний.

Покрываемые вопросы были сформулированы непосредственно по разделам
клинических рекомендаций. Поэтому эта выборка представляет контролируемую
проверку RAG и не полностью отражает распределение естественных
пользовательских медицинских запросов.

Оценка качества ответов и их обоснованности выполнялась LLM-оценщиком
по заранее заданным рубрикам, а не независимым медицинским экспертом.
Поэтому результаты показывают сравнительное поведение вариантов системы,
но не являются клинической валидацией.

Слепая оценка также имела техническое ограничение: в отдельных ответах
могли сохраняться альтернативные маркеры источников. Для известного
затронутого примера был проведён отдельный анализ устойчивости,
и основной результат после его исключения практически не изменился.

Сравнение B и C оценивает RAG-конфигурацию целиком. Вариант C отличается
от B не только наличием найденного контекста, но и дополнительными
инструкциями по использованию источников. Поэтому наблюдаемый эффект
нельзя приписывать только retrieval.

Для вопросов вне покрытия проверялось, достаточно ли информации
в retrieved top-3. Недостаточность этих трёх фрагментов не доказывает
полное отсутствие любой связанной информации во всей базе знаний.

Наконец, эта отложенная выборка относится только к текущему RAG-этапу.
После просмотра её результатов она не используется для настройки
следующих вариантов системы. Финальная проектная test-выборка должна
оставаться неизменной до фиксации всех экспериментальных вариантов.

## Вывод

На данном отложенном benchmark вариант C с RAG заметно чаще получал
более высокую оценку, чем baseline, когда найденный контекст содержал
релевантную информацию.

При этом RAG сам по себе не предотвращает неподтверждённую генерацию.

Даже при хорошем retrieval модель может:

- неправильно интерпретировать найденный источник;
- изменить направление или силу клинической рекомендации;
- добавить неподтверждённые медицинские утверждения;
- неправильно связать утверждение со ссылкой;
- использовать ближайший, но нерелевантный источник, когда найденной
  информации недостаточно.

Поэтому качество RAG зависит как минимум от трёх компонентов:

1. качества retrieval;
2. корректности интерпретации найденной информации при генерации;
3. способности системы определить, что найденного контекста недостаточно.

Retrieval показал высокий recall, однако корректное использование источников
и безопасное поведение при недостаточном контексте остаются существенными
источниками ошибок.

Эти результаты показывают, что RAG может улучшать качество ответов
на вопросах, хорошо покрываемых базой знаний, но для более надёжной
медицинской системы необходим отдельный контроль достаточности контекста,
корректности ссылок и соответствия сгенерированных утверждений
найденным источникам.